# NB Expresson Hackathon

## Pipeline

Notebook นี้ไหลจากข้อมูลดิบไปสู่ submission ตามลำดับด้านล่าง

```mermaid
flowchart LR
    A["Load data + data contract"] --> B["Build daily target panel"]
    B --> C["Parse sample rows + decision dates"]
    C --> D["Feature registry"]
    D --> E["Known-future features"]
    D --> F["As-of behavioral features"]
    E --> G["Model frame validation"]
    G --> H["Backtest folds"]
    H --> I["AutoGluon + CatBoost"]
    I --> J["Blend + calibration + cap"]
    J --> K["Final train + prediction"]
    K --> L["submission.csv"]
```

| ช่วงงาน | หน้าที่ |
|---|---|
| Data contract | โหลดตารางและตรวจไฟล์ต้องห้าม |
| Target panel | รวม `units_sold` เป็นรายวันระดับร้านและหมวดสินค้า |
| Feature frame | รวม static, known-future และ as-of features |
| Validation | ใช้ blind folds เพื่อเลือก blend/postprocess |
| Final export | train ด้วยประวัติทั้งหมดแล้วเขียน `submission.csv` |

## ขั้นที่ 0: เตรียม Environment และ Library


In [ ]:
!pip -q install autogluon.timeseries "torch<2.10" torchvision torchaudio lxml beautifulsoup4 catboost scikit-learn

In [ ]:
# ส่วนนี้เตรียม library และ runtime ให้พร้อมก่อนเริ่ม pipeline ทั้งหมด
# นำเข้า standard library ที่ใช้จัดการ package, path, subprocess และไฟล์ zip
import importlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# ตั้งค่า runtime ให้ใช้จำนวน worker เหมาะสมและควบคุม fallback install ผ่าน environment variable
os.environ.setdefault('LOKY_MAX_CPU_COUNT', '4')
INSTALL_MISSING_PACKAGES = os.environ.get('INSTALL_MISSING_PACKAGES', '1') == '1'

# โหลด library หลักสำหรับคำนวณเชิงตัวเลข จัดการ dataframe และวัด MAE
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

# โหลด AutoGluon TimeSeries; ถ้าไม่มีและอนุญาตไว้ จะติดตั้งอัตโนมัติ
try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
except Exception:
    if not INSTALL_MISSING_PACKAGES:
        raise
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-q',
        'autogluon.timeseries', 'torch<2.10', 'torchvision', 'torchaudio',
    ])
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# โหลด CatBoost สำหรับ tabular model ที่ใช้เป็น candidate ใน ensemble
try:
    from catboost import CatBoostRegressor
except Exception:
    if not INSTALL_MISSING_PACKAGES:
        raise
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
    from catboost import CatBoostRegressor

# ตั้งค่าการแสดงผลและ seed เพื่อให้อ่านผลใน notebook ง่ายและ reproducible ขึ้น
pd.set_option('display.max_columns', 180)
pd.set_option('display.width', 200)

SEED = 42
np.random.seed(SEED)

## ขั้นที่ 1: ตั้งค่าโจทย์ Forecast และ Hyperparameters

In [ ]:
# ส่วนนี้รวมค่าคงที่ของโจทย์และ hyperparameters ที่ใช้ทั้ง notebook
# กำหนด path หลักของ workspace และตำแหน่งข้อมูล
WORKSPACE_DIR = Path.cwd()
COMPETITION_DATA_DIR = WORKSPACE_DIR / 'super-ai-engineer-season-6-coffee-chain-hackathon'
TRAIN_TABLE_DIR = COMPETITION_DATA_DIR / 'train'
TEST_TABLE_DIR = COMPETITION_DATA_DIR / 'test'
TEACHING_ARTIFACT_DIR = WORKSPACE_DIR / 'outputs' / 'knowledge_share_teaching'
# กำหนดช่วง train/future และความยาว prediction horizon
SUBMISSION_START_DATE = pd.Timestamp('2024-11-01')
SUBMISSION_END_DATE = pd.Timestamp('2024-12-31')
HISTORY_END_DATE = pd.Timestamp('2024-10-31')
FEATURE_START = pd.Timestamp('2023-01-01')
FORECAST_WINDOW_DAYS = int((SUBMISSION_END_DATE - SUBMISSION_START_DATE).days + 1)

# กำหนด mapping ของ horizon และลำดับ category/day-of-week ที่ใช้ cast categorical
HORIZON_DAY_LOOKUP = {'1d': 1, '7d': 7, '1m': 30}
CATEGORY_ORDER = [
    'Coffee', 'Tea', 'Bakery', 'Savory Bakery',
    'Chocolate & Milk', 'Juice & Smoothie', 'Merchandise',
]
WEEKDAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
EPS = 1e-6

# กำหนด flag สำหรับควบคุม runtime หนัก เช่น CV และ final AutoGluon training
ENABLE_BACKTEST = True
RUN_FINAL_AUTOGLUON_TRAINING = True

# กำหนดเวลา train และค่าค้นหา blend/calibration/cap
AUTOGLUON_CV_TIME_LIMIT_SECONDS = 1000
AUTOGLUON_FINAL_TIME_LIMIT_SECONDS = 3000
BLEND_RANDOM_SEARCH_ITER = 2000
CALIBRATION_SHRINK_N = 600
CAP_TUNE_GRID_MULTS = [1.05, 1.15, 1.25, 1.35, 1.50]
CATBOOST_ITERATIONS_CV = 700
CATBOOST_ITERATIONS_FINAL = 1300

# กำหนด blind validation folds ตาม cutoff ที่ใช้ประเมินย้อนหลัง
BACKTEST_WINDOWS = [
    {'name': 'Sep_blind', 'cutoff': pd.Timestamp('2024-08-31'), 'valid_start': pd.Timestamp('2024-09-01'), 'valid_end': pd.Timestamp('2024-09-30')},
    {'name': 'Oct_blind', 'cutoff': pd.Timestamp('2024-09-30'), 'valid_start': pd.Timestamp('2024-10-01'), 'valid_end': pd.Timestamp('2024-10-31')},
]

# กำหนด model family ของ AutoGluon ที่ใช้ใน time-series predictor
AUTOGLUON_MODEL_CONFIG = {
    'DirectTabular': {},
    'Chronos': {
        'ag_args': {'name_suffix': 'WithRegressor'},
        'model_path': 'bolt_small',
        'target_scaler': 'standard',
        'covariate_regressor': {'model_name': 'CAT', 'model_hyperparameters': {'iterations': 1000}},
    },
}

# สร้าง run id และ path ของ model artifacts โดยยังไม่สร้าง directory จนกว่าจะ train จริง
TEACHING_RUN_ID = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
MODEL_ARTIFACT_DIR = TEACHING_ARTIFACT_DIR / f'teaching_models_{TEACHING_RUN_ID}'
print({'prediction_length': FORECAST_WINDOW_DAYS, 'model_root': str(MODEL_ARTIFACT_DIR)})

## ขั้นที่ 2: โหลดข้อมูลและตรวจสอบ Data Contract

In [ ]:
# ส่วนนี้โหลดข้อมูลจากไฟล์จริงและตรวจเงื่อนไขเพื่อป้องกัน leakage ก่อนเริ่มสร้าง target
# ระบุแหล่ง fallback และรายชื่อไฟล์ที่จำเป็นสำหรับการแข่งขัน
GOOGLE_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1ztRWsCT8YTAQOtFtREEXB-Ny2HNGOx7r'
REQUIRED_TRAIN_TABLES = {
    'product': 'PRODUCT.csv',
    'store': 'STORE.csv',
    'date_dim': 'DATE_DIM.csv',
    'order': 'ORDER.csv',
    'transaction': 'TRANSACTION.csv',
    'inventory': 'INVENTORY.csv',
    'promotion': 'PROMOTION.csv',
    'local_event': 'LOCAL_EVENT.csv',
}
SAMPLE_FILE = 'sample_submission_with_id.csv'


def discover_dataset_paths():
    """รวบรวม path ของไฟล์ข้อมูล train และ sample ที่ต้องใช้ พร้อมระบุไฟล์ที่ยังขาด เพื่อให้โหลดข้อมูลได้อย่างมี contract ชัดเจน"""
    train_paths = {name: TRAIN_TABLE_DIR / filename for name, filename in REQUIRED_TRAIN_TABLES.items()}
    sample_path = COMPETITION_DATA_DIR / SAMPLE_FILE
    missing = [str(path) for path in list(train_paths.values()) + [sample_path] if not path.exists()]
    return {'train': train_paths, 'sample': sample_path, 'missing': missing}


def _import_gdown():
    """นำเข้า gdown สำหรับดาวน์โหลดข้อมูลจาก Google Drive และติดตั้งแบบ fallback เมื่ออนุญาตให้ติดตั้ง package อัตโนมัติ"""
    try:
        return importlib.import_module('gdown')
    except Exception:
        if not INSTALL_MISSING_PACKAGES:
            raise
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
        return importlib.import_module('gdown')


def _table_target_from_columns(csv_path):
    """ตรวจ schema ของไฟล์ CSV ที่ดาวน์โหลดมา แล้วจับคู่กับชื่อไฟล์มาตรฐานที่ notebook ต้องใช้"""
    try:
        columns = set(pd.read_csv(csv_path, nrows=0).columns)
    except Exception:
        return None
    signatures = [
        ({'product_id', 'product_name', 'category', 'serve_type', 'base_price'}, TRAIN_TABLE_DIR / 'PRODUCT.csv'),
        ({'store_id', 'neighborhood_type', 'seating_capacity', 'has_drive_through', 'staff_count', 'opened_date'}, TRAIN_TABLE_DIR / 'STORE.csv'),
        ({'date', 'day_of_week', 'week_number', 'month', 'quarter', 'is_weekend'}, TRAIN_TABLE_DIR / 'DATE_DIM.csv'),
        ({'order_id', 'store_id', 'date', 'hour', 'customer_id', 'payment_method'}, TRAIN_TABLE_DIR / 'ORDER.csv'),
        ({'transaction_id', 'order_id', 'product_id', 'units_sold', 'discount_applied', 'revenue'}, TRAIN_TABLE_DIR / 'TRANSACTION.csv'),
        ({'inventory_id', 'store_id', 'product_id', 'date', 'opening_stock', 'closing_stock', 'is_stockout'}, TRAIN_TABLE_DIR / 'INVENTORY.csv'),
        ({'promo_id', 'campaign_id', 'product_id', 'store_id', 'start_date', 'end_date', 'promo_type'}, TRAIN_TABLE_DIR / 'PROMOTION.csv'),
        ({'event_id', 'store_id', 'date', 'event_name', 'event_type'}, TRAIN_TABLE_DIR / 'LOCAL_EVENT.csv'),
    ]
    for required_columns, destination in signatures:
        if required_columns.issubset(columns):
            return destination
    if 'id' in columns and len(columns) <= 3:
        return COMPETITION_DATA_DIR / SAMPLE_FILE
    return None


def _normalize_downloaded_tables(raw_dir):
    """คัดลอกไฟล์ CSV ที่ดาวน์โหลดมาไปยังตำแหน่ง train/sample ที่ถูกต้องตาม schema ของแต่ละตาราง"""
    COMPETITION_DATA_DIR.mkdir(parents=True, exist_ok=True)
    TRAIN_TABLE_DIR.mkdir(parents=True, exist_ok=True)
    copied = {}
    for csv_path in sorted(Path(raw_dir).rglob('*.csv')):
        target = _table_target_from_columns(csv_path)
        if target is None:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(csv_path, target)
        copied[target.name] = str(csv_path)
    return copied


def _prepare_dataset_if_needed():
    """เตรียม dataset เมื่อไฟล์ยังไม่ครบ โดยลองแตก zip หรือดาวน์โหลด fallback ก่อนหยุดด้วย error ที่ชัดเจน"""
    paths = discover_dataset_paths()
    if not paths['missing']:
        return
    zip_candidates = [
        WORKSPACE_DIR / 'super-ai-engineer-season-6-coffee-chain-hackathon.zip',
        Path('/content/super-ai-engineer-season-6-coffee-chain-hackathon.zip'),
        Path('/content/drive/MyDrive/super-ai-engineer-season-6-coffee-chain-hackathon.zip'),
    ]
    zip_path = next((path for path in zip_candidates if path.exists()), None)
    if zip_path is not None:
        print('Extracting dataset zip:', zip_path)
        COMPETITION_DATA_DIR.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(COMPETITION_DATA_DIR)
    if discover_dataset_paths()['missing']:
        raw_dir = COMPETITION_DATA_DIR / '_teaching_drive_raw'
        if raw_dir.exists():
            shutil.rmtree(raw_dir)
        raw_dir.mkdir(parents=True, exist_ok=True)
        print('Downloading public teaching dataset folder:', GOOGLE_DRIVE_FOLDER_URL)
        _import_gdown().download_folder(GOOGLE_DRIVE_FOLDER_URL, output=str(raw_dir), quiet=False, use_cookies=False)
        print('Recognized tables:', _normalize_downloaded_tables(raw_dir))
    remaining = discover_dataset_paths()['missing']
    if remaining:
        raise FileNotFoundError(f'Missing required dataset files: {remaining}')


def normalize_table_dates(table_map):
    """แปลงคอลัมน์วันที่ในทุกตารางให้เป็น datetime เพื่อให้ join, filter และ feature engineering ทำงานถูกต้อง"""
    out = {name: df.copy() for name, df in table_map.items()}
    out['store']['opened_date'] = pd.to_datetime(out['store']['opened_date'].astype(str).str.strip(), errors='coerce')
    out['date_dim']['date'] = pd.to_datetime(out['date_dim']['date'])
    out['order']['date'] = pd.to_datetime(out['order']['date'])
    out['inventory']['date'] = pd.to_datetime(out['inventory']['date'])
    out['promotion']['start_date'] = pd.to_datetime(out['promotion']['start_date'].astype(str).str.strip(), errors='coerce')
    out['promotion']['end_date'] = pd.to_datetime(out['promotion']['end_date'].astype(str).str.strip(), errors='coerce')
    out['local_event']['date'] = pd.to_datetime(out['local_event']['date'])
    return out


# ตรวจ forbidden test tables ก่อนโหลดข้อมูล เพื่อหยุดทันทีถ้าเจอข้อมูลอนาคตที่ไม่ควรใช้
def assert_no_forbidden_test_tables():
    """ตรวจว่าไม่มี operational test tables ที่อาจทำให้เกิด future leakage ปะปนอยู่ใน workspace"""
    forbidden_paths = [TEST_TABLE_DIR / name for name in ['TRANSACTION.csv', 'ORDER.csv', 'INVENTORY.csv']]
    found = [str(path) for path in forbidden_paths if path.exists()]
    if found:
        raise RuntimeError(f'Forbidden future operational tables found: {found}')


def load_competition_tables():
    """โหลดตารางการแข่งขันทั้งหมดหลังผ่านการเตรียมไฟล์และ normalize วันที่แล้ว"""
    _prepare_dataset_if_needed()
    paths = discover_dataset_paths()
    loaded = {name: pd.read_csv(path) for name, path in paths['train'].items()}
    loaded['sample'] = pd.read_csv(paths['sample'])
    return normalize_table_dates(loaded)


assert_no_forbidden_test_tables()
# โหลด tables ทั้งหมดเข้าสู่ memory หลังผ่านการตรวจไฟล์และ normalize date
tables = load_competition_tables()
# แตก table map ออกเป็นตัวแปรชื่อสั้น เพื่อให้ cell ถัดไปอ่านง่ายขึ้น
product_table = tables['product']
store_table = tables['store']
calendar_table = tables['date_dim']
order_table = tables['order']
transaction_table = tables['transaction']
inventory_table = tables['inventory']
promotion_table = tables['promotion']
local_event_table = tables['local_event']
submission_template = tables['sample']

# แสดงขนาดของแต่ละตารางเพื่อ sanity check ว่าโหลดข้อมูลครบ
print({name: df.shape for name, df in tables.items()})

## ขั้นที่ 3: สร้าง Target

In [ ]:
# ส่วนนี้แปลง transaction-level data เป็น daily target panel สำหรับ training
def attach_order_context_to_transactions(transaction_df, order_df):
    """เติมบริบทคำสั่งซื้อ เช่นร้าน วันที่ ชั่วโมง และลูกค้า ให้ transaction แต่ละแถว"""
    order_columns = ['order_id', 'store_id', 'date', 'hour', 'customer_id', 'is_member']
    return transaction_df.merge(order_df[order_columns], on='order_id', how='left', validate='many_to_one')


def attach_product_context_to_transactions(transaction_order_df, product_df):
    """เติมบริบทสินค้าและ category ให้ transaction เพื่อใช้ aggregate target ตาม store/category/date"""
    product_columns = ['product_id', 'category', 'base_price']
    return transaction_order_df.merge(product_df[product_columns], on='product_id', how='left', validate='many_to_one')


def summarize_daily_demand(enriched_transactions):
    """รวมยอด units_sold รายวันในระดับ store_id และ category เพื่อสร้าง target หลักของโจทย์"""
    daily = (
        enriched_transactions
        .groupby(['store_id', 'category', 'date'], observed=True, as_index=False)['units_sold']
        .sum()
        .rename(columns={'date': 'timestamp'})
    )
    if not np.isclose(enriched_transactions['units_sold'].sum(), daily['units_sold'].sum()):
        raise AssertionError('Units changed during daily aggregation')
    return daily


def make_active_store_category_grid(store_df, categories, start, end):
    """สร้าง calendar grid ของร้านและ category เฉพาะวันที่ร้านเปิดใช้งานแล้ว เพื่อเติมวันที่ไม่มียอดขายเป็นศูนย์"""
    grid = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, pd.date_range(start, end, freq='D')],
        names=['store_id', 'category', 'timestamp'],
    ).to_frame(index=False)
    grid = grid.merge(store_df[['store_id', 'opened_date']], on='store_id', how='left', validate='many_to_one')
    grid['effective_opened_date'] = grid['opened_date'].clip(lower=pd.Timestamp(start))
    active = grid[grid['timestamp'].ge(grid['effective_opened_date'])].copy()
    active['is_active_train_day'] = True
    return active


def build_training_target_panel(order_df, transaction_df, product_df, store_df):
    """สร้าง target panel สำหรับ training จาก transaction จริง พร้อมเติมวันขาดและสร้าง item_id สำหรับ time-series model"""
    txn_with_order = attach_order_context_to_transactions(transaction_df, order_df)
    txn_prod_local = attach_product_context_to_transactions(txn_with_order, product_df)
    daily_target = summarize_daily_demand(txn_prod_local)
    active_grid = make_active_store_category_grid(store_df, CATEGORY_ORDER, FEATURE_START, HISTORY_END_DATE)
    panel = active_grid.merge(daily_target, on=['store_id', 'category', 'timestamp'], how='left')
    panel['units_sold'] = panel['units_sold'].fillna(0)
    category_slug = panel['category'].astype(str).str.replace(r'[^A-Za-z0-9]+', '-', regex=True).str.strip('-')
    panel['item_id'] = panel['store_id'].astype(str) + '_' + category_slug
    panel = panel[['store_id', 'category', 'timestamp', 'units_sold', 'opened_date', 'effective_opened_date', 'is_active_train_day', 'item_id']]
    if panel.duplicated(['store_id', 'category', 'timestamp']).any():
        raise AssertionError('Duplicate target rows after active calendar completion')
    return panel


def make_horizon_item_id(store_id, category, horizon):
    """สร้าง item_id ที่รวม store, category และ horizon เพื่อแยก series ตามระยะ forecast"""
    category_slug = pd.Series(category).astype(str).str.replace(r'[^A-Za-z0-9]+', '-', regex=True).str.strip('-')
    return pd.Series(store_id).astype(str) + '_' + category_slug + '_' + pd.Series(horizon).astype(str)


# สร้าง training_target_panel จริงจาก order/transaction/product/store แล้วตรวจ shape เบื้องต้น
# training_target_panel = build_training_target_panel(order_table, transaction_table, product_table, store_table)
# แสดงจำนวนแถว target panel เพื่อยืนยันว่า grid และ active-store filter ทำงาน
# print(training_target_panel.shape)
# training_target_panel.head()

## ขั้นที่ 4: Features

In [ ]:
# ส่วนนี้นิยาม feature contract ทั้งหมดที่ model frame ต้องมี
# รายชื่อ static features ที่ไม่เปลี่ยนตาม timestamp และใช้แนบเป็น static metadata
STATIC_FEATURE_COLS = [
    'store_id', 'category', 'horizon', 'horizon_days', 'neighborhood_type',
    'seating_capacity', 'has_drive_through', 'staff_count', 'open_hour', 'close_hour', 'operating_hours',
    'capacity_per_staff', 'category_avg_base_price', 'category_min_base_price', 'category_max_base_price',
    'category_price_range', 'category_product_count', 'limited_edition_product_share',
]

# รายชื่อ calendar และ decision-date features ที่รู้ล่วงหน้าได้
CALENDAR_FEATURES = [
    'day_of_week', 'dow_num', 'day_of_month', 'week_number', 'week_of_month', 'month', 'quarter',
    'is_weekend', 'is_holiday', 'is_school_break', 'is_payday', 'is_rainy_season',
    'sin_doy', 'cos_doy', 'is_month_start', 'is_month_end', 'days_to_month_end', 'days_from_month_start',
    'days_to_next_holiday', 'days_since_prev_holiday', 'pre_holiday_3d', 'post_holiday_3d',
    'decision_dow_num', 'decision_day_of_month', 'decision_month', 'decision_is_weekend', 'decision_is_holiday',
    'decision_is_payday', 'days_from_decision_to_forecast', 'is_decision_clipped_to_train_end',
    'effective_days_from_decision_to_forecast',
]
# รายชื่อ promotion features ที่สรุปจาก campaign/product/store/date
PROMO_FEATURES = [
    'active_promo_products', 'max_discount_pct', 'mean_discount_pct', 'promo_type_count', 'email_sent',
    'social_campaign', 'campaign_count', 'promo_duration_mean', 'days_since_promo_start_min',
    'days_until_promo_end_min', 'promo_weighted_discount_by_price', 'promo_coverage_ratio',
]
# รายชื่อ event density features ที่บอกความหนาแน่นของ local events รอบร้าน
LOCAL_EVENT_DENSITY_FEATURES = [
    'local_event_count', 'local_event_type_count', 'event_count_next_1d', 'event_count_next_3d',
    'event_count_next_7d', 'event_count_prev_3d', 'days_until_next_local_event', 'event_cluster_week',
]
# รายชื่อ sales as-of features ที่ต้องคำนวณจากอดีตเท่านั้น
SALES_ASOF_FEATURES = [
    'decision_sales_lag_1', 'decision_sales_lag_7', 'decision_sales_lag_14', 'decision_sales_lag_28',
    'decision_sales_mean_7', 'decision_sales_mean_14', 'decision_sales_mean_28', 'decision_sales_mean_56',
    'decision_sales_median_28', 'decision_sales_std_28', 'decision_sales_max_28', 'decision_sales_q25_28',
    'decision_sales_q75_28', 'same_dow_mean_4w', 'same_dow_mean_8w', 'decision_sales_mean_7_over_28',
]
# รายชื่อ momentum/share features เพื่อจับทิศทางยอดขายระดับร้านและ category
MOMENTUM_FEATURES = [
    'store_total_sales_mean_7', 'store_total_sales_mean_28', 'store_total_sales_trend_28',
    'category_chain_sales_mean_7', 'category_chain_sales_mean_28', 'category_chain_sales_trend_28',
    'store_category_share_28', 'category_share_change_7_vs_28',
]
# รายชื่อ order/customer behavior features จากข้อมูลคำสั่งซื้อย้อนหลัง
ORDER_CUSTOMER_FEATURES = [
    'order_count_mean_7', 'order_count_mean_28', 'unique_customer_count_28', 'known_customer_share_28',
    'member_order_share_28', 'avg_units_per_order_28', 'avg_order_hour_28', 'morning_order_share_28',
    'evening_order_share_28', 'payment_method_entropy_28',
]
# รายชื่อ revenue และ discount features จาก transaction history
REVENUE_DISCOUNT_FEATURES = [
    'revenue_sum_28', 'revenue_per_unit_28', 'avg_transaction_revenue_28', 'discount_applied_mean_28',
    'discount_applied_share_28', 'category_revenue_share_28',
]
# รายชื่อ inventory/stockout features ที่ใช้บริบท stock จากอดีต
INVENTORY_FEATURES = [
    'stockout_rate_7', 'stockout_rate_28', 'stockout_sku_share_28', 'opening_stock_mean_7',
    'units_received_sum_7', 'closing_stock_mean_7', 'days_since_last_stockout', 'stock_cover_proxy_28',
]
# รายชื่อ lifecycle/weather/event relevance features ที่เสริมบริบทของร้านและวัน forecast
STORE_LIFECYCLE_FEATURES = ['store_age_days', 'store_age_months', 'store_age_log', 'is_new_store_365d']
WEATHER_FEATURES = ['temperature_2m_mean', 'temperature_2m_min', 'rain_flag', 'humid_day_flag']
SPECIAL_EVENT_FEATURES = ['special_event_count', 'special_event_category_match', 'special_event_store_weight', 'special_event_score']
EVENT_RELEVANCE_FEATURES = ['event_store_relevance_max', 'event_category_relevance_max', 'event_relevance_score', 'event_intensity_max']
EVENT_KEYWORD_FEATURES = [
    'event_kw_food_category_score', 'event_kw_music_category_score', 'event_kw_student_category_score',
    'event_kw_book_category_score', 'event_kw_auto_category_score', 'event_kw_night_category_score',
]

# รวม feature lists เป็น group map เพื่อให้สร้าง registry และ audit ได้ง่าย
FEATURE_GROUPS = {
    'static_store_category_product': STATIC_FEATURE_COLS,
    'calendar_decision': CALENDAR_FEATURES,
    'promotion_economics': PROMO_FEATURES,
    'local_event_density': LOCAL_EVENT_DENSITY_FEATURES,
    'asof_sales_behavior': SALES_ASOF_FEATURES,
    'store_category_momentum': MOMENTUM_FEATURES,
    'order_customer_behavior': ORDER_CUSTOMER_FEATURES,
    'revenue_discount_behavior': REVENUE_DISCOUNT_FEATURES,
    'inventory_stockout_coverage': INVENTORY_FEATURES,
    'store_lifecycle': STORE_LIFECYCLE_FEATURES,
    'weather': WEATHER_FEATURES,
    'special_event': SPECIAL_EVENT_FEATURES,
    'event_relevance_weights': EVENT_RELEVANCE_FEATURES,
    'event_keyword_category_score': EVENT_KEYWORD_FEATURES,
}


def make_feature_registry(group_map):
    """สร้างตาราง metadata ของ feature ทุกตัว เพื่อบอกกลุ่ม แหล่งข้อมูล availability และวิธีเติม missing"""
    source_map = {
        'static_store_category_product': 'STORE/PRODUCT/sample horizon',
        'calendar_decision': 'DATE_DIM and decision dates',
        'promotion_economics': 'PROMOTION/PRODUCT',
        'local_event_density': 'LOCAL_EVENT',
        'asof_sales_behavior': 'target panel as-of',
        'store_category_momentum': 'target panel as-of',
        'order_customer_behavior': 'ORDER/TRANSACTION as-of',
        'revenue_discount_behavior': 'TRANSACTION revenue as-of',
        'inventory_stockout_coverage': 'INVENTORY as-of',
        'store_lifecycle': 'STORE opened_date',
        'weather': 'Bangkok climatology',
        'special_event': 'curated public events',
        'event_relevance_weights': 'event type relevance map',
        'event_keyword_category_score': 'event keyword relevance map',
    }
    rows = []
    for group, features in group_map.items():
        for feature in features:
            rows.append({
                'feature': feature,
                'group': group,
                'source': source_map[group],
                'availability': 'static' if group == 'static_store_category_product' else ('as_of' if 'behavior' in group or group == 'inventory_stockout_coverage' else 'known_future'),
                'is_static': group == 'static_store_category_product',
                'fill_value': 'missing' if feature in {'category', 'horizon', 'neighborhood_type', 'day_of_week'} else 0,
            })
    return pd.DataFrame(rows)


# สร้าง registry แล้ว derive feature lists ที่ downstream ใช้จริง
FEATURE_REGISTRY = make_feature_registry(FEATURE_GROUPS)
STATIC_FEATURE_COLS = FEATURE_REGISTRY.loc[FEATURE_REGISTRY['is_static'], 'feature'].tolist()
future_known_feature_cols = FEATURE_REGISTRY.loc[~FEATURE_REGISTRY['is_static'], 'feature'].tolist()
model_feature_cols = STATIC_FEATURE_COLS + future_known_feature_cols


# ตรวจจำนวน feature แต่ละกลุ่มและ assert contract 18/121/139
print('Feature groups:')
for name, cols in FEATURE_GROUPS.items():
    print(f'  {name}: {len(cols)}')
print('Static features:', len(STATIC_FEATURE_COLS))
print('Known covariates:', len(future_known_feature_cols))
print('Total features:', len(STATIC_FEATURE_COLS) + len(future_known_feature_cols))
assert len(STATIC_FEATURE_COLS) == 18
assert len(future_known_feature_cols) == 121
assert len(STATIC_FEATURE_COLS) + len(future_known_feature_cols) == 139

## ขั้นที่ 5: สร้าง Features

In [ ]:
# ส่วนนี้สร้าง features ที่รู้ได้ล่วงหน้าหรือไม่พึ่งยอดขายอนาคต
# กำหนดช่วง feature calendar ให้ครอบคลุมทั้ง history และ forecast period
FEATURE_START = pd.Timestamp('2023-01-01')
EPS = 1e-6



# กลุ่มฟังก์ชันนี้ parse sample submission และสร้าง decision-date fields
def parse_submission_key_parts(sample_df):
    """แยก id ใน sample submission ให้เป็น store_id, category, forecast_date และ horizon ที่ใช้สร้าง prediction rows"""
    pattern = r'^(?P<store_id>\d+)_(?P<category>.+)_(?P<forecast_date>\d{4}-\d{2}-\d{2})_(?P<horizon>1d|7d|1m)$'
    parts = sample_df['id'].str.extract(pattern)
    failed = parts.isna().any(axis=1)
    if failed.any():
        raise ValueError(f'Could not parse submission ids: {sample_df.loc[failed, "id"].head(10).tolist()}')
    parts['store_id'] = parts['store_id'].astype(int)
    parts['forecast_date'] = pd.to_datetime(parts['forecast_date'])
    return pd.concat([sample_df[['id']].reset_index(drop=True), parts.reset_index(drop=True)], axis=1)


def add_horizon_fields(parsed_df):
    """เติม horizon_days จากชื่อ horizon เพื่อใช้คำนวณ decision date ของแต่ละแถว forecast"""
    out = parsed_df.copy()
    out['horizon_days'] = out['horizon'].map(HORIZON_DAY_LOOKUP).astype(int)
    return out


def add_decision_dates(horizon_df, history_cutoff=HISTORY_END_DATE):
    """คำนวณ decision_date และ effective_decision_date เพื่อจำกัดข้อมูลที่ feature มองเห็นตาม cutoff"""
    out = horizon_df.copy()
    out['decision_date'] = out['forecast_date'] - pd.to_timedelta(out['horizon_days'], unit='D')
    cutoff = pd.Timestamp(history_cutoff)
    out['effective_decision_date'] = out['decision_date'].where(out['decision_date'].le(cutoff), cutoff)
    return out


def make_submission_target_rows(sample_df):
    """แปลง sample submission เป็น target rows ที่มี key ครบสำหรับ join กับ forecast_feature_frame"""
    parsed = parse_submission_key_parts(sample_df)
    with_horizon = add_horizon_fields(parsed)
    with_decision = add_decision_dates(with_horizon)
    with_decision['item_id'] = make_horizon_item_id(with_decision['store_id'], with_decision['category'], with_decision['horizon'])
    return with_decision


# กลุ่มฟังก์ชันนี้สร้าง static features จาก STORE และ PRODUCT
def add_store_static_features(store_df):
    """สร้าง static feature ของร้าน เช่น capacity, staff, operating hours และ drive-through flag"""
    out = store_df.copy()
    out['open_hour'] = pd.to_numeric(out['open_time'].astype(str).str[:2], errors='coerce')
    out['close_hour'] = pd.to_numeric(out['close_time'].astype(str).str[:2], errors='coerce')
    out['operating_hours'] = (out['close_hour'] - out['open_hour']).clip(lower=0)
    out['capacity_per_staff'] = out['seating_capacity'] / out['staff_count'].replace(0, np.nan)
    out['capacity_per_staff'] = out['capacity_per_staff'].replace([np.inf, -np.inf], np.nan).fillna(0)
    out['has_drive_through'] = out['has_drive_through'].fillna(False).astype(int)
    return out


def build_category_static_features(product_df):
    """สรุป feature คงที่ของ category จากตาราง product เช่นราคาเฉลี่ย ราคา min/max และจำนวนสินค้า"""
    out = (
        product_df.groupby('category', observed=True)
        .agg(
            category_avg_base_price=('base_price', 'mean'),
            category_min_base_price=('base_price', 'min'),
            category_max_base_price=('base_price', 'max'),
            category_product_count=('product_id', 'nunique'),
            limited_edition_product_share=('is_limited_edition', 'mean'),
        )
        .reset_index()
    )
    out['category_price_range'] = out['category_max_base_price'] - out['category_min_base_price']
    return out


# กลุ่มฟังก์ชันนี้สร้าง forecast index และ calendar features
def create_forecast_index_frame(store_df, categories, horizons, start, end, history_cutoff):
    """สร้าง index หลักของทุก store/category/horizon/date ที่ต้องมีใน train, validation และ future frame"""
    dates = pd.date_range(start, end, freq='D')
    full = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, list(horizons.keys()), dates],
        names=['store_id', 'category', 'horizon', 'timestamp'],
    ).to_frame(index=False)
    full['horizon_days'] = full['horizon'].map(horizons).astype(int)
    full['decision_date'] = full['timestamp'] - pd.to_timedelta(full['horizon_days'], unit='D')
    cutoff = pd.Timestamp(history_cutoff)
    full['effective_decision_date'] = full['decision_date'].where(full['decision_date'].le(cutoff), cutoff)
    full['item_id'] = make_horizon_item_id(full['store_id'], full['category'], full['horizon'])
    return full


def holiday_distance_features(dates, holiday_dates):
    """คำนวณระยะห่างจากวันหยุดก่อนหน้าและถัดไป เพื่อใช้เป็น calendar signal"""
    day_arr = pd.Series(pd.to_datetime(dates)).values.astype('datetime64[D]')
    holiday_arr = np.array(sorted(pd.to_datetime(holiday_dates).unique()), dtype='datetime64[D]')
    if len(holiday_arr) == 0:
        prev_days = np.full(len(day_arr), 99)
        next_days = np.full(len(day_arr), 99)
    else:
        prev_idx = np.searchsorted(holiday_arr, day_arr, side='right') - 1
        next_idx = np.searchsorted(holiday_arr, day_arr, side='left')
        prev_days = np.where(
            prev_idx >= 0,
            (day_arr - holiday_arr[np.clip(prev_idx, 0, len(holiday_arr) - 1)]).astype('timedelta64[D]').astype(int),
            99,
        )
        next_days = np.where(
            next_idx < len(holiday_arr),
            (holiday_arr[np.clip(next_idx, 0, len(holiday_arr) - 1)] - day_arr).astype('timedelta64[D]').astype(int),
            99,
        )
    return np.clip(prev_days, 0, 99), np.clip(next_days, 0, 99)


def build_calendar_lookup(date_dim_df, start, end):
    """สร้างตาราง calendar feature รายวัน เช่น day-of-week, payday, month boundary และ holiday distance"""
    cal = pd.DataFrame({'timestamp': pd.date_range(start, end, freq='D')})
    dim = date_dim_df.rename(columns={'date': 'timestamp'}).copy()
    cal = cal.merge(dim, on='timestamp', how='left')
    cal['day_of_week'] = cal['day_of_week'].fillna(cal['timestamp'].dt.day_name())
    cal['day_of_week'] = pd.Categorical(cal['day_of_week'], WEEKDAY_ORDER, ordered=True)
    cal['dow_num'] = cal['timestamp'].dt.dayofweek
    cal['day_of_month'] = cal['timestamp'].dt.day
    if 'week_number' not in cal.columns:
        cal['week_number'] = cal['timestamp'].dt.isocalendar().week.astype(int)
    cal['week_number'] = pd.to_numeric(cal['week_number'], errors='coerce').fillna(cal['timestamp'].dt.isocalendar().week.astype(int)).astype(int)
    cal['week_of_month'] = ((cal['day_of_month'] - 1) // 7 + 1).astype(int)
    cal['month'] = pd.to_numeric(cal['month'], errors='coerce').fillna(cal['timestamp'].dt.month).astype(int)
    cal['quarter'] = pd.to_numeric(cal['quarter'], errors='coerce').fillna(cal['timestamp'].dt.quarter).astype(int)
    cal['is_weekend'] = cal['is_weekend'].fillna(cal['dow_num'].ge(5)).astype(int)
    for col in ['is_holiday', 'is_school_break', 'is_payday', 'is_rainy_season']:
        cal[col] = cal[col].fillna(False).astype(int)
    cal['sin_doy'] = np.sin(2 * np.pi * cal['timestamp'].dt.dayofyear / 365.25)
    cal['cos_doy'] = np.cos(2 * np.pi * cal['timestamp'].dt.dayofyear / 365.25)
    cal['is_month_start'] = cal['timestamp'].dt.is_month_start.astype(int)
    cal['is_month_end'] = cal['timestamp'].dt.is_month_end.astype(int)
    cal['days_to_month_end'] = (cal['timestamp'].dt.days_in_month - cal['day_of_month']).astype(int)
    cal['days_from_month_start'] = (cal['day_of_month'] - 1).astype(int)
    holiday_dates = cal.loc[cal['is_holiday'].eq(1), 'timestamp']
    prev_holiday, next_holiday = holiday_distance_features(cal['timestamp'], holiday_dates)
    cal['days_since_prev_holiday'] = prev_holiday
    cal['days_to_next_holiday'] = next_holiday
    cal['pre_holiday_3d'] = cal['days_to_next_holiday'].between(0, 3).astype(int)
    cal['post_holiday_3d'] = cal['days_since_prev_holiday'].between(0, 3).astype(int)
    return cal[CALENDAR_FEATURES[:22] + ['timestamp']]


def add_calendar_features(frame, calendar_lookup):
    """merge calendar features ของ forecast timestamp เข้ากับ frame หลัก"""
    return frame.merge(calendar_lookup, on='timestamp', how='left')


def attach_decision_calendar_features(frame, calendar_lookup):
    """เติม calendar features ของ effective_decision_date เพื่อให้โมเดลรู้บริบทของวันตัดสินใจ"""
    decision_cols = ['timestamp', 'dow_num', 'day_of_month', 'month', 'is_weekend', 'is_holiday', 'is_payday']
    decision = calendar_lookup[decision_cols].rename(columns={
        'timestamp': 'effective_decision_date',
        'dow_num': 'decision_dow_num',
        'day_of_month': 'decision_day_of_month',
        'month': 'decision_month',
        'is_weekend': 'decision_is_weekend',
        'is_holiday': 'decision_is_holiday',
        'is_payday': 'decision_is_payday',
    })
    out = frame.merge(decision, on='effective_decision_date', how='left')
    out['days_from_decision_to_forecast'] = (out['timestamp'] - out['decision_date']).dt.days.clip(lower=0)
    out['is_decision_clipped_to_train_end'] = out['decision_date'].gt(pd.Timestamp(HISTORY_END_DATE)).astype(int)
    out['effective_days_from_decision_to_forecast'] = (out['timestamp'] - out['effective_decision_date']).dt.days.clip(lower=0)
    for col in ['decision_dow_num', 'decision_day_of_month', 'decision_month', 'decision_is_weekend', 'decision_is_holiday', 'decision_is_payday']:
        out[col] = pd.to_numeric(out[col], errors='coerce').fillna(0).astype(int)
    return out


# กลุ่มฟังก์ชันนี้ขยายและสรุป promotion เป็น daily features
def expand_promotion_calendar(promo_df, product_df):
    """ขยาย promotion จากช่วง start/end date เป็นแถวรายวัน พร้อมตัดแถววันที่ผิดรูปหรือไม่สมบูรณ์ออก"""
    promo_prod = promo_df.merge(product_df[['product_id', 'category', 'base_price']], on='product_id', how='left', validate='many_to_one')
    valid = promo_prod['start_date'].notna() & promo_prod['end_date'].notna() & promo_prod['end_date'].ge(promo_prod['start_date'])
    pieces = []
    for row in promo_prod.loc[valid].itertuples(index=False):
        days = pd.date_range(row.start_date, row.end_date, freq='D')
        duration = (pd.Timestamp(row.end_date) - pd.Timestamp(row.start_date)).days + 1
        part = pd.DataFrame({
            'store_id': row.store_id,
            'category': row.category,
            'timestamp': days,
            'product_id': row.product_id,
            'campaign_id': row.campaign_id,
            'promo_type': row.promo_type,
            'discount_pct': row.discount_pct,
            'email_sent': int(pd.notna(row.email_sent) and bool(row.email_sent)),
            'social_campaign': int(pd.notna(row.social_campaign) and bool(row.social_campaign)),
            'base_price': row.base_price,
            'promo_duration': duration,
        })
        part['days_since_promo_start'] = (part['timestamp'] - pd.Timestamp(row.start_date)).dt.days
        part['days_until_promo_end'] = (pd.Timestamp(row.end_date) - part['timestamp']).dt.days
        part['discount_price_weight'] = part['discount_pct'] * part['base_price']
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True) if pieces else pd.DataFrame()


def summarize_promotion_calendar(daily, product_df):
    """สรุป promotion รายวันเป็น feature ต่อ store/category/date เช่นจำนวนสินค้า promo และ discount intensity"""
    if daily.empty:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'] + PROMO_FEATURES)
    out = (
        daily.groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            active_promo_products=('product_id', 'nunique'),
            max_discount_pct=('discount_pct', 'max'),
            mean_discount_pct=('discount_pct', 'mean'),
            promo_type_count=('promo_type', 'nunique'),
            email_sent=('email_sent', 'max'),
            social_campaign=('social_campaign', 'max'),
            campaign_count=('campaign_id', 'nunique'),
            promo_duration_mean=('promo_duration', 'mean'),
            days_since_promo_start_min=('days_since_promo_start', 'min'),
            days_until_promo_end_min=('days_until_promo_end', 'min'),
            discount_price_weight_sum=('discount_price_weight', 'sum'),
            promo_base_price_sum=('base_price', 'sum'),
        )
        .reset_index()
    )
    out['promo_weighted_discount_by_price'] = out['discount_price_weight_sum'] / out['promo_base_price_sum'].replace(0, np.nan)
    product_counts = product_df.groupby('category', observed=True)['product_id'].nunique().rename('category_product_count').reset_index()
    out = out.merge(product_counts, on='category', how='left', validate='many_to_one')
    out['promo_coverage_ratio'] = out['active_promo_products'] / out['category_product_count'].replace(0, np.nan)
    return out[['store_id', 'category', 'timestamp'] + PROMO_FEATURES]


def make_promotion_daily_features(promo_df, product_df):
    """สร้าง promotion feature table รายวันโดยขยาย promotion calendar แล้ว aggregate กลับเป็นระดับโมเดล"""
    return summarize_promotion_calendar(expand_promotion_calendar(promo_df, product_df), product_df)


# กลุ่มฟังก์ชันนี้สร้าง local-event, special-event และ weather known covariates
def normalize_local_event_type(value):
    """ทำความสะอาดชื่อ event type ให้เป็นรูปแบบ lowercase underscore ที่ใช้ lookup weight ได้สม่ำเสมอ"""
    return str(value).strip().lower().replace(' ', '_')


def prepare_local_event_table(local_event_df):
    """เตรียม local event table ด้วย timestamp และ normalized event type ก่อนนำไปสร้าง event features"""
    events = local_event_df.rename(columns={'date': 'timestamp'}).copy()
    events['timestamp'] = pd.to_datetime(events['timestamp'])
    events['event_type_norm'] = events['event_type'].map(normalize_local_event_type)
    return events


def build_local_event_density_features(local_event_df, store_df, start, end):
    """สร้าง feature ความหนาแน่นของ event รอบร้าน เช่น event count ย้อนหลัง/ล่วงหน้า และระยะถึง event ถัดไป"""
    events = prepare_local_event_table(local_event_df)
    base = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), pd.date_range(start, end, freq='D')],
        names=['store_id', 'timestamp'],
    ).to_frame(index=False)
    raw = (
        events.groupby(['store_id', 'timestamp'], observed=True)
        .agg(local_event_count=('event_id', 'count'), local_event_type_count=('event_type_norm', 'nunique'))
        .reset_index()
    )
    out = base.merge(raw, on=['store_id', 'timestamp'], how='left')
    out[['local_event_count', 'local_event_type_count']] = out[['local_event_count', 'local_event_type_count']].fillna(0)
    out = out.sort_values(['store_id', 'timestamp']).reset_index(drop=True)
    pieces = []
    for _, part in out.groupby('store_id', observed=True, sort=False):
        part = part.copy()
        s = part['local_event_count'].astype(float)
        part['event_count_next_1d'] = s.iloc[::-1].rolling(1, min_periods=1).sum().iloc[::-1].values
        part['event_count_next_3d'] = s.iloc[::-1].rolling(3, min_periods=1).sum().iloc[::-1].values
        part['event_count_next_7d'] = s.iloc[::-1].rolling(7, min_periods=1).sum().iloc[::-1].values
        part['event_count_prev_3d'] = s.shift(1).rolling(3, min_periods=1).sum().fillna(0).values
        event_dates = part.loc[part['local_event_count'].gt(0), 'timestamp'].to_numpy(dtype='datetime64[D]')
        days = part['timestamp'].to_numpy(dtype='datetime64[D]')
        if len(event_dates) == 0:
            next_days = np.full(len(part), 99)
        else:
            next_idx = np.searchsorted(event_dates, days, side='left')
            next_days = np.where(
                next_idx < len(event_dates),
                (event_dates[np.clip(next_idx, 0, len(event_dates) - 1)] - days).astype('timedelta64[D]').astype(int),
                99,
            )
        part['days_until_next_local_event'] = np.clip(next_days, 0, 99)
        part['event_cluster_week'] = part['event_count_next_7d'].ge(3).astype(int)
        pieces.append(part)
    return pd.concat(pieces, ignore_index=True)[['store_id', 'timestamp'] + LOCAL_EVENT_DENSITY_FEATURES]


# ตาราง weight เหล่านี้เป็น assumption สำหรับแปลง event type เป็นคะแนน relevance
EVENT_TYPE_STORE_RELEVANCE = {
    'concert': {'mall': 1.0, 'tourist': 0.9, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'music_festival': {'mall': 1.0, 'tourist': 1.0, 'transit': 0.8, 'urban_residential': 0.7, 'university': 0.7, 'office': 0.4, 'hospital': 0.3, 'gas_station': 0.3},
    'food_festival': {'mall': 0.9, 'tourist': 1.0, 'urban_residential': 0.8, 'transit': 0.7, 'university': 0.6, 'office': 0.5, 'hospital': 0.4, 'gas_station': 0.4},
    'market': {'urban_residential': 1.0, 'tourist': 0.8, 'transit': 0.7, 'university': 0.7, 'mall': 0.5, 'office': 0.4, 'hospital': 0.4, 'gas_station': 0.4},
    'book_fair': {'university': 1.0, 'mall': 0.8, 'tourist': 0.6, 'urban_residential': 0.6, 'office': 0.5, 'transit': 0.5, 'hospital': 0.3, 'gas_station': 0.2},
    'convention': {'office': 1.0, 'mall': 0.9, 'transit': 0.8, 'tourist': 0.7, 'university': 0.7, 'urban_residential': 0.5, 'hospital': 0.4, 'gas_station': 0.3},
    'sports': {'university': 0.9, 'urban_residential': 0.8, 'transit': 0.7, 'tourist': 0.6, 'mall': 0.5, 'gas_station': 0.5, 'office': 0.3, 'hospital': 0.3},
    'cultural': {'tourist': 1.0, 'urban_residential': 0.8, 'mall': 0.7, 'university': 0.6, 'transit': 0.6, 'gas_station': 0.4, 'hospital': 0.3, 'office': 0.3},
}
EVENT_TYPE_CATEGORY_RELEVANCE = {
    'concert': {'Coffee': 1.0, 'Tea': 0.9, 'Chocolate & Milk': 0.8, 'Juice & Smoothie': 0.7, 'Bakery': 0.6, 'Savory Bakery': 0.6, 'Merchandise': 0.2},
    'music_festival': {'Coffee': 1.0, 'Tea': 0.9, 'Chocolate & Milk': 0.8, 'Juice & Smoothie': 0.8, 'Bakery': 0.6, 'Savory Bakery': 0.7, 'Merchandise': 0.2},
    'food_festival': {'Savory Bakery': 1.0, 'Bakery': 0.9, 'Juice & Smoothie': 0.8, 'Coffee': 0.7, 'Tea': 0.6, 'Chocolate & Milk': 0.6, 'Merchandise': 0.1},
    'market': {'Coffee': 0.8, 'Tea': 0.7, 'Bakery': 0.8, 'Savory Bakery': 0.8, 'Juice & Smoothie': 0.7, 'Chocolate & Milk': 0.5, 'Merchandise': 0.2},
    'book_fair': {'Coffee': 1.0, 'Tea': 0.8, 'Bakery': 0.6, 'Chocolate & Milk': 0.5, 'Savory Bakery': 0.4, 'Juice & Smoothie': 0.4, 'Merchandise': 0.3},
    'convention': {'Coffee': 1.0, 'Tea': 0.7, 'Savory Bakery': 0.8, 'Bakery': 0.6, 'Chocolate & Milk': 0.4, 'Juice & Smoothie': 0.3, 'Merchandise': 0.2},
    'sports': {'Juice & Smoothie': 1.0, 'Coffee': 0.8, 'Tea': 0.7, 'Savory Bakery': 0.5, 'Bakery': 0.4, 'Chocolate & Milk': 0.4, 'Merchandise': 0.1},
    'cultural': {'Coffee': 0.8, 'Tea': 0.8, 'Bakery': 0.7, 'Juice & Smoothie': 0.7, 'Chocolate & Milk': 0.5, 'Savory Bakery': 0.5, 'Merchandise': 0.2},
}
EVENT_TYPE_BASE_INTENSITY = {
    'music_festival': 1.20, 'concert': 1.10, 'food_festival': 1.05, 'convention': 0.95,
    'market': 0.85, 'book_fair': 0.80, 'sports': 0.75, 'cultural': 0.70,
}
EVENT_KEYWORDS = {
    'food': ['อาหาร', 'street food', 'ของหวาน', 'เบียร์', 'คราฟท์', 'food', 'dessert'],
    'music': ['ดนตรี', 'คอนเสิร์ต', 'แจ๊ส', 'busking', 'music', 'concert', 'festival'],
    'student': ['นักศึกษา', 'startup', 'job fair', 'กีฬาสี'],
    'book': ['หนังสือ', 'book'],
    'auto': ['auto', 'motor', 'รถ'],
    'night': ['night', 'กลางคืน', 'busking', 'แจ๊ส'],
}


def extract_event_keyword_flags(event_name):
    """ตรวจ keyword ในชื่อ event เพื่อแปลงเป็น flag ของธีม event ที่เกี่ยวกับ category บางกลุ่ม"""
    text = str(event_name).lower()
    return {key: int(any(token.lower() in text for token in tokens)) for key, tokens in EVENT_KEYWORDS.items()}


def build_event_relevance_features(local_event_df, store_df, categories):
    """แปลง local events เป็นคะแนน relevance ต่อ store/category/date โดยใช้ event type, neighborhood และ keyword weights"""
    events = prepare_local_event_table(local_event_df).merge(store_df[['store_id', 'neighborhood_type']], on='store_id', how='left')
    rows = []
    for event in events.to_dict('records'):
        event_type = event['event_type_norm']
        store_type = event['neighborhood_type']
        base_intensity = EVENT_TYPE_BASE_INTENSITY.get(event_type, 0.50)
        store_weight = EVENT_TYPE_STORE_RELEVANCE.get(event_type, {}).get(store_type, 0.35)
        kw_flags = extract_event_keyword_flags(event['event_name'])
        for category in categories:
            cat_weight = EVENT_TYPE_CATEGORY_RELEVANCE.get(event_type, {}).get(category, 0.35)
            row = {
                'store_id': event['store_id'],
                'category': category,
                'timestamp': event['timestamp'],
                'event_store_relevance_max': store_weight,
                'event_category_relevance_max': cat_weight,
                'event_relevance_score': base_intensity * store_weight * cat_weight,
                'event_intensity_max': base_intensity,
            }
            for key, flag in kw_flags.items():
                row[f'event_kw_{key}_category_score'] = flag * base_intensity * store_weight * cat_weight
            rows.append(row)
    if not rows:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'] + EVENT_RELEVANCE_FEATURES + EVENT_KEYWORD_FEATURES)
    raw = pd.DataFrame(rows)
    agg = {col: 'sum' for col in ['event_relevance_score'] + EVENT_KEYWORD_FEATURES if col in raw.columns}
    agg.update({'event_store_relevance_max': 'max', 'event_category_relevance_max': 'max', 'event_intensity_max': 'max'})
    out = raw.groupby(['store_id', 'category', 'timestamp'], observed=True).agg(agg).reset_index()
    return out[['store_id', 'category', 'timestamp'] + EVENT_RELEVANCE_FEATURES + EVENT_KEYWORD_FEATURES]


def make_curated_event_calendar():
    """สร้าง calendar ของ special events ที่รู้ล่วงหน้า เพื่อใช้เป็น known-future event signal"""
    rows = [
        ('awakening_bangkok_2024', 'Awakening Bangkok', '2024-11-08', '2024-11-17', ['Coffee', 'Tea'], ['tourist', 'urban_residential'], ['mall', 'transit'], 0.90),
        ('loy_krathong_2024', 'Loy Krathong', '2024-11-15', '2024-11-15', ['Juice & Smoothie', 'Bakery'], ['tourist', 'urban_residential', 'mall'], ['transit', 'university'], 1.00),
        ('motor_expo_2024', 'Motor Expo 2024', '2024-11-29', '2024-12-10', ['Savory Bakery', 'Coffee'], ['gas_station', 'transit'], ['tourist', 'urban_residential', 'mall'], 0.95),
        ('red_cross_fair_2024', 'Red Cross Fair', '2024-12-11', '2024-12-22', ['ALL'], ['tourist', 'urban_residential', 'mall'], ['transit', 'office', 'university', 'hospital'], 1.15),
        ('christmas_2024', 'Christmas', '2024-12-25', '2024-12-25', ['Merchandise', 'Chocolate & Milk'], ['mall', 'tourist', 'urban_residential'], ['office', 'transit'], 1.00),
        ('new_year_countdown_2024', 'New Year Countdown', '2024-12-31', '2024-12-31', ['ALL'], ['tourist', 'mall', 'transit', 'urban_residential'], ['gas_station', 'office'], 1.20),
        ('graduation_university_proxy_2024', 'University graduation proxy', '2024-10-01', '2024-10-31', ['Coffee', 'Bakery', 'Merchandise'], ['university'], ['transit', 'mall'], 0.50),
    ]
    expanded = []
    for event_key, event_name, start, end, categories, strong_types, secondary_types, base_weight in rows:
        for ts in pd.date_range(start, end, freq='D'):
            expanded.append({
                'event_key': event_key,
                'event_name': event_name,
                'timestamp': ts,
                'impacted_categories': categories,
                'strong_store_types': strong_types,
                'secondary_store_types': secondary_types,
                'base_weight': base_weight,
            })
    return pd.DataFrame(expanded)


def make_special_event_features(calendar_df, store_df, categories):
    """แปลง special event calendar เป็น feature ต่อ store/category/date พร้อมคะแนนความเกี่ยวข้องของร้านและ category"""
    rows = []
    for event in calendar_df.to_dict('records'):
        for store_row in store_df[['store_id', 'neighborhood_type']].to_dict('records'):
            stype = store_row['neighborhood_type']
            if 'ALL' in event['strong_store_types'] or stype in event['strong_store_types']:
                store_weight = 1.0
            elif 'ALL' in event['secondary_store_types'] or stype in event['secondary_store_types']:
                store_weight = 0.65
            else:
                store_weight = 0.25
            for category in categories:
                category_match = 'ALL' in event['impacted_categories'] or category in event['impacted_categories']
                rows.append({
                    'store_id': store_row['store_id'],
                    'category': category,
                    'timestamp': event['timestamp'],
                    'special_event_count': 1,
                    'special_event_category_match': int(category_match),
                    'special_event_store_weight': store_weight,
                    'special_event_score': float(event['base_weight']) * store_weight if category_match else 0.0,
                })
    if not rows:
        return pd.DataFrame(columns=['store_id', 'category', 'timestamp'] + SPECIAL_EVENT_FEATURES)
    return (
        pd.DataFrame(rows)
        .groupby(['store_id', 'category', 'timestamp'], observed=True)
        .agg(
            special_event_count=('special_event_count', 'sum'),
            special_event_category_match=('special_event_category_match', 'max'),
            special_event_store_weight=('special_event_store_weight', 'max'),
            special_event_score=('special_event_score', 'sum'),
        )
        .reset_index()
    )


def build_weather_features(start, end):
    """สร้าง weather climatology รายวันจากค่าเฉลี่ยรายเดือน เพื่อเป็น known covariate ที่ไม่ใช้ข้อมูลอนาคตรายวันจริง"""
    dates = pd.date_range(start, end, freq='D')
    normals = pd.DataFrame({
        'month': list(range(1, 13)),
        'temperature_2m_mean': [26.7, 28.2, 29.7, 30.7, 30.1, 29.5, 29.1, 28.9, 28.6, 28.2, 27.5, 26.3],
        'temperature_2m_min': [21.8, 23.4, 25.0, 26.1, 26.0, 25.8, 25.5, 25.4, 25.1, 24.6, 23.4, 21.7],
        'relative_humidity_2m_mean': [66, 69, 72, 73, 76, 77, 78, 79, 81, 79, 73, 67],
        'precipitation_sum': [9, 20, 40, 91, 248, 157, 175, 219, 333, 190, 40, 11],
    })
    out = pd.DataFrame({'timestamp': dates})
    out['month'] = out['timestamp'].dt.month
    out = out.merge(normals, on='month', how='left').drop(columns=['month'])
    out['precipitation_sum'] = out['precipitation_sum'] / out['timestamp'].dt.days_in_month
    out['rain_flag'] = out['precipitation_sum'].gt(0).astype(int)
    out['humid_day_flag'] = out['relative_humidity_2m_mean'].ge(80).astype(int)
    out = out[['timestamp'] + WEATHER_FEATURES]
    return out

## ขั้นที่ 6: สร้าง Behavioral Features

In [ ]:
# ส่วนนี้สร้าง behavioral features จากอดีตทั้งหมดแบบ as-of เพื่อกัน leakage
# กลุ่มฟังก์ชันนี้สร้าง daily base grid และ rolling helper ที่ใช้ร่วมกัน
def make_target_daily_full(target_panel_df, store_df, categories, start, end):
    """สร้าง target daily grid เต็มช่วงเวลาในระดับ store/category/effective_decision_date สำหรับคำนวณ as-of features"""
    idx = pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, pd.date_range(start, end, freq='D')],
        names=['store_id', 'category', 'effective_decision_date'],
    ).to_frame(index=False)
    target = target_panel_df.rename(columns={'timestamp': 'effective_decision_date'})[
        ['store_id', 'category', 'effective_decision_date', 'units_sold']
    ]
    out = idx.merge(target, on=['store_id', 'category', 'effective_decision_date'], how='left')
    out['units_sold'] = pd.to_numeric(out['units_sold'], errors='coerce').fillna(0)
    return out.sort_values(['store_id', 'category', 'effective_decision_date']).reset_index(drop=True)


def shifted_rolling_by_group(df, group_cols, value_col, window, agg='mean', min_periods=1):
    """คำนวณ rolling statistic แบบ leakage-safe โดย shift ข้อมูลหนึ่งวันก่อน rolling ทุกครั้ง"""
    grouped = df.groupby(group_cols, observed=True, sort=False)[value_col]
    if agg == 'mean':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).mean())
    if agg == 'sum':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).sum())
    if agg == 'median':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).median())
    if agg == 'std':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=2).std())
    if agg == 'max':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).max())
    if agg == 'q25':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).quantile(0.25))
    if agg == 'q75':
        return grouped.transform(lambda s: s.shift(1).rolling(window, min_periods=min_periods).quantile(0.75))
    raise ValueError(agg)


# กลุ่มฟังก์ชันนี้สร้าง sales lag/rolling และ momentum features
def build_sales_asof_features(target_daily):
    """สร้าง lag และ rolling sales features จาก target panel โดยใช้ข้อมูลที่เกิดก่อน decision date เท่านั้น"""
    out = target_daily.copy()
    group_cols = ['store_id', 'category']
    grouped = out.groupby(group_cols, observed=True, sort=False)['units_sold']
    for lag in [1, 7, 14, 28]:
        out[f'decision_sales_lag_{lag}'] = grouped.shift(lag)
    for window in [7, 14, 28, 56]:
        out[f'decision_sales_mean_{window}'] = shifted_rolling_by_group(out, group_cols, 'units_sold', window, 'mean')
    out['decision_sales_median_28'] = shifted_rolling_by_group(out, group_cols, 'units_sold', 28, 'median')
    out['decision_sales_std_28'] = shifted_rolling_by_group(out, group_cols, 'units_sold', 28, 'std')
    out['decision_sales_max_28'] = shifted_rolling_by_group(out, group_cols, 'units_sold', 28, 'max')
    out['decision_sales_q25_28'] = shifted_rolling_by_group(out, group_cols, 'units_sold', 28, 'q25')
    out['decision_sales_q75_28'] = shifted_rolling_by_group(out, group_cols, 'units_sold', 28, 'q75')
    out['dow_num'] = out['effective_decision_date'].dt.dayofweek
    out['same_dow_mean_4w'] = out.groupby(group_cols + ['dow_num'], observed=True, sort=False)['units_sold'].transform(
        lambda s: s.shift(1).rolling(4, min_periods=1).mean()
    )
    out['same_dow_mean_8w'] = out.groupby(group_cols + ['dow_num'], observed=True, sort=False)['units_sold'].transform(
        lambda s: s.shift(1).rolling(8, min_periods=1).mean()
    )
    out['decision_sales_mean_7_over_28'] = out['decision_sales_mean_7'] / (out['decision_sales_mean_28'] + EPS)
    out[SALES_ASOF_FEATURES] = out[SALES_ASOF_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out[['store_id', 'category', 'effective_decision_date'] + SALES_ASOF_FEATURES]


def build_momentum_features(target_daily):
    """สร้าง trend และ share features ของยอดขายระดับร้านและ category เพื่อจับ momentum ระยะสั้นเทียบระยะยาว"""
    base = target_daily[['store_id', 'category', 'effective_decision_date', 'units_sold']].copy()
    store_daily = base.groupby(['store_id', 'effective_decision_date'], observed=True)['units_sold'].sum().reset_index()
    store_daily['store_total_sales_mean_7'] = shifted_rolling_by_group(store_daily, ['store_id'], 'units_sold', 7, 'mean')
    store_daily['store_total_sales_mean_28'] = shifted_rolling_by_group(store_daily, ['store_id'], 'units_sold', 28, 'mean')
    store_daily['store_total_sum_7'] = shifted_rolling_by_group(store_daily, ['store_id'], 'units_sold', 7, 'sum')
    store_daily['store_total_sum_28'] = shifted_rolling_by_group(store_daily, ['store_id'], 'units_sold', 28, 'sum')
    store_daily['store_total_sales_trend_28'] = store_daily['store_total_sales_mean_7'] / (store_daily['store_total_sales_mean_28'] + EPS)

    chain_cat_daily = base.groupby(['category', 'effective_decision_date'], observed=True)['units_sold'].sum().reset_index()
    chain_cat_daily['category_chain_sales_mean_7'] = shifted_rolling_by_group(chain_cat_daily, ['category'], 'units_sold', 7, 'mean')
    chain_cat_daily['category_chain_sales_mean_28'] = shifted_rolling_by_group(chain_cat_daily, ['category'], 'units_sold', 28, 'mean')
    chain_cat_daily['category_chain_sales_trend_28'] = chain_cat_daily['category_chain_sales_mean_7'] / (chain_cat_daily['category_chain_sales_mean_28'] + EPS)

    base['store_category_sum_7'] = shifted_rolling_by_group(base, ['store_id', 'category'], 'units_sold', 7, 'sum')
    base['store_category_sum_28'] = shifted_rolling_by_group(base, ['store_id', 'category'], 'units_sold', 28, 'sum')
    out = base.merge(
        store_daily[['store_id', 'effective_decision_date', 'store_total_sales_mean_7', 'store_total_sales_mean_28', 'store_total_sales_trend_28', 'store_total_sum_7', 'store_total_sum_28']],
        on=['store_id', 'effective_decision_date'],
        how='left',
    )
    out = out.merge(
        chain_cat_daily[['category', 'effective_decision_date', 'category_chain_sales_mean_7', 'category_chain_sales_mean_28', 'category_chain_sales_trend_28']],
        on=['category', 'effective_decision_date'],
        how='left',
    )
    share_7 = out['store_category_sum_7'] / (out['store_total_sum_7'] + EPS)
    out['store_category_share_28'] = out['store_category_sum_28'] / (out['store_total_sum_28'] + EPS)
    out['category_share_change_7_vs_28'] = share_7 - out['store_category_share_28']
    out[MOMENTUM_FEATURES] = out[MOMENTUM_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out[['store_id', 'category', 'effective_decision_date'] + MOMENTUM_FEATURES]


# กลุ่มฟังก์ชันนี้สร้าง order/customer behavior features จากข้อมูลคำสั่งซื้อย้อนหลัง
def complete_daily_feature_frame(store_df, categories, start, end):
    """สร้าง daily grid มาตรฐานของ store/category/effective_decision_date เพื่อใช้เป็นฐานของ as-of feature tables"""
    return pd.MultiIndex.from_product(
        [sorted(store_df['store_id'].unique()), categories, pd.date_range(start, end, freq='D')],
        names=['store_id', 'category', 'effective_decision_date'],
    ).to_frame(index=False)


def build_order_category_rows(order_df, transaction_df, product_df):
    """รวม transaction, order และ product ให้เป็นแถวระดับ order/category พร้อม flag พฤติกรรมคำสั่งซื้อ"""
    txn_order = transaction_df.merge(
        order_df[['order_id', 'store_id', 'date', 'hour', 'customer_id', 'is_member', 'payment_method']],
        on='order_id',
        how='left',
        validate='many_to_one',
    ).merge(product_df[['product_id', 'category']], on='product_id', how='left', validate='many_to_one')
    order_cat = (
        txn_order.groupby(['store_id', 'category', 'date', 'order_id'], observed=True, dropna=False)
        .agg(
            hour=('hour', 'first'),
            customer_id=('customer_id', 'first'),
            is_member=('is_member', 'first'),
            payment_method=('payment_method', 'first'),
            units_in_order=('units_sold', 'sum'),
        )
        .reset_index()
    )
    order_cat['known_customer_order'] = order_cat['customer_id'].notna().astype(int)
    order_cat['member_order'] = order_cat['is_member'].fillna(False).astype(int)
    order_cat['morning_order'] = order_cat['hour'].between(6, 11).astype(int)
    order_cat['evening_order'] = order_cat['hour'].ge(17).astype(int)
    return order_cat


def summarize_daily_order_counts(order_cat):
    """สรุป order/category rows เป็น daily order behavior เช่นจำนวน order, member orders และช่วงเวลาการซื้อ"""
    return (
        order_cat.groupby(['store_id', 'category', 'date'], observed=True)
        .agg(
            order_count=('order_id', 'nunique'),
            unique_customer_count=('customer_id', 'nunique'),
            known_customer_orders=('known_customer_order', 'sum'),
            member_orders=('member_order', 'sum'),
            units_sum=('units_in_order', 'sum'),
            hour_sum=('hour', 'sum'),
            morning_orders=('morning_order', 'sum'),
            evening_orders=('evening_order', 'sum'),
        )
        .reset_index()
        .rename(columns={'date': 'effective_decision_date'})
    )


def summarize_payment_method_counts(order_cat):
    """นับจำนวน order ตาม payment method ต่อวัน เพื่อใช้คำนวณ payment mix และ entropy"""
    pay_counts = (
        order_cat.pivot_table(
            index=['store_id', 'category', 'date'],
            columns='payment_method',
            values='order_id',
            aggfunc='nunique',
            fill_value=0,
            observed=True,
        )
        .reset_index()
        .rename(columns={'date': 'effective_decision_date'})
    )
    pay_cols = [c for c in pay_counts.columns if c not in ['store_id', 'category', 'effective_decision_date']]
    renamed = pay_counts.rename(columns={c: f'pay_count_{i}' for i, c in enumerate(pay_cols)})
    return renamed, [c for c in renamed.columns if c.startswith('pay_count_')]


def add_payment_entropy_feature(out, group_cols, pay_count_cols):
    """คำนวณ entropy ของ payment method ย้อนหลัง 28 วัน เพื่อวัดความหลากหลายของช่องทางชำระเงิน"""
    if not pay_count_cols:
        out['payment_method_entropy_28'] = 0.0
        return out
    rolling_pay = [shifted_rolling_by_group(out, group_cols, col, 28, 'sum').fillna(0).to_numpy() for col in pay_count_cols]
    mat = np.vstack(rolling_pay).T
    row_sum = mat.sum(axis=1, keepdims=True)
    p = np.divide(mat, row_sum, out=np.zeros_like(mat, dtype=float), where=row_sum > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        entropy_terms = np.where(p > 0, p * np.log(p), 0.0)
    out['payment_method_entropy_28'] = -entropy_terms.sum(axis=1)
    return out


def build_order_customer_features(order_df, transaction_df, product_df, store_df, categories, start, end):
    """สร้าง feature พฤติกรรม order และ customer แบบ as-of จากข้อมูลคำสั่งซื้อย้อนหลัง"""
    order_cat = build_order_category_rows(order_df, transaction_df, product_df)
    daily = summarize_daily_order_counts(order_cat)
    pay_counts, pay_count_cols = summarize_payment_method_counts(order_cat)
    keys = ['store_id', 'category', 'effective_decision_date']
    base = complete_daily_feature_frame(store_df, categories, start, end)
    out = base.merge(daily, on=keys, how='left', validate='one_to_one')
    out = out.merge(pay_counts, on=keys, how='left', validate='one_to_one')
    raw_cols = ['order_count', 'unique_customer_count', 'known_customer_orders', 'member_orders', 'units_sum', 'hour_sum', 'morning_orders', 'evening_orders'] + pay_count_cols
    out[raw_cols] = out[raw_cols].fillna(0)
    group_cols = ['store_id', 'category']
    out['order_count_sum_28'] = shifted_rolling_by_group(out, group_cols, 'order_count', 28, 'sum')
    out['order_count_mean_7'] = shifted_rolling_by_group(out, group_cols, 'order_count', 7, 'mean')
    out['order_count_mean_28'] = shifted_rolling_by_group(out, group_cols, 'order_count', 28, 'mean')
    out['unique_customer_count_28'] = shifted_rolling_by_group(out, group_cols, 'unique_customer_count', 28, 'sum')
    known_sum_28 = shifted_rolling_by_group(out, group_cols, 'known_customer_orders', 28, 'sum')
    member_sum_28 = shifted_rolling_by_group(out, group_cols, 'member_orders', 28, 'sum')
    units_sum_28 = shifted_rolling_by_group(out, group_cols, 'units_sum', 28, 'sum')
    hour_sum_28 = shifted_rolling_by_group(out, group_cols, 'hour_sum', 28, 'sum')
    morning_sum_28 = shifted_rolling_by_group(out, group_cols, 'morning_orders', 28, 'sum')
    evening_sum_28 = shifted_rolling_by_group(out, group_cols, 'evening_orders', 28, 'sum')
    denom = out['order_count_sum_28'] + EPS
    out['known_customer_share_28'] = known_sum_28 / denom
    out['member_order_share_28'] = member_sum_28 / denom
    out['avg_units_per_order_28'] = units_sum_28 / denom
    out['avg_order_hour_28'] = hour_sum_28 / denom
    out['morning_order_share_28'] = morning_sum_28 / denom
    out['evening_order_share_28'] = evening_sum_28 / denom
    out = add_payment_entropy_feature(out, group_cols, pay_count_cols)
    out[ORDER_CUSTOMER_FEATURES] = out[ORDER_CUSTOMER_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out[['store_id', 'category', 'effective_decision_date'] + ORDER_CUSTOMER_FEATURES]


# กลุ่มฟังก์ชันนี้สร้าง revenue/discount features จาก transaction history
def build_revenue_discount_features(order_df, transaction_df, product_df, store_df, categories, start, end):
    """สร้าง feature รายได้และ discount แบบ rolling เช่น revenue per unit และ discount share จากข้อมูลอดีต"""
    txn = transaction_df.merge(order_df[['order_id', 'store_id', 'date']], on='order_id', how='left').merge(product_df[['product_id', 'category']], on='product_id', how='left')
    txn['discount_positive'] = pd.to_numeric(txn['discount_applied'], errors='coerce').fillna(0).gt(0).astype(int)
    daily = (
        txn.groupby(['store_id', 'category', 'date'], observed=True)
        .agg(
            revenue_daily=('revenue', 'sum'),
            units_daily=('units_sold', 'sum'),
            transaction_count=('transaction_id', 'count'),
            discount_sum=('discount_applied', 'sum'),
            discount_positive_count=('discount_positive', 'sum'),
        )
        .reset_index()
        .rename(columns={'date': 'effective_decision_date'})
    )
    base = complete_daily_feature_frame(store_df, categories, start, end)
    out = base.merge(daily, on=['store_id', 'category', 'effective_decision_date'], how='left')
    raw_cols = ['revenue_daily', 'units_daily', 'transaction_count', 'discount_sum', 'discount_positive_count']
    out[raw_cols] = out[raw_cols].fillna(0)
    group_cols = ['store_id', 'category']
    revenue_sum_28 = shifted_rolling_by_group(out, group_cols, 'revenue_daily', 28, 'sum')
    units_sum_28 = shifted_rolling_by_group(out, group_cols, 'units_daily', 28, 'sum')
    txn_count_28 = shifted_rolling_by_group(out, group_cols, 'transaction_count', 28, 'sum')
    discount_sum_28 = shifted_rolling_by_group(out, group_cols, 'discount_sum', 28, 'sum')
    discount_positive_28 = shifted_rolling_by_group(out, group_cols, 'discount_positive_count', 28, 'sum')
    out['revenue_sum_28'] = revenue_sum_28
    out['revenue_per_unit_28'] = revenue_sum_28 / (units_sum_28 + EPS)
    out['avg_transaction_revenue_28'] = revenue_sum_28 / (txn_count_28 + EPS)
    out['discount_applied_mean_28'] = discount_sum_28 / (txn_count_28 + EPS)
    out['discount_applied_share_28'] = discount_positive_28 / (txn_count_28 + EPS)
    store_rev = out.groupby(['store_id', 'effective_decision_date'], observed=True)['revenue_daily'].sum().reset_index()
    store_rev['store_revenue_sum_28'] = shifted_rolling_by_group(store_rev, ['store_id'], 'revenue_daily', 28, 'sum')
    out = out.merge(store_rev[['store_id', 'effective_decision_date', 'store_revenue_sum_28']], on=['store_id', 'effective_decision_date'], how='left')
    out['category_revenue_share_28'] = out['revenue_sum_28'] / (out['store_revenue_sum_28'] + EPS)
    out[REVENUE_DISCOUNT_FEATURES] = out[REVENUE_DISCOUNT_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out[['store_id', 'category', 'effective_decision_date'] + REVENUE_DISCOUNT_FEATURES]


# กลุ่มฟังก์ชันนี้สร้าง inventory และ stockout features จากอดีต
def build_inventory_features(inventory_df, product_df, store_df, categories, start, end):
    """สร้าง inventory และ stockout features แบบ as-of เช่น stockout rate, stock received และ days since stockout"""
    inv = inventory_df.merge(product_df[['product_id', 'category']], on='product_id', how='left')
    daily = (
        inv.groupby(['store_id', 'category', 'date'], observed=True)
        .agg(
            any_stockout=('is_stockout', 'max'),
            stockout_sku_share_daily=('is_stockout', 'mean'),
            opening_stock_daily=('opening_stock', 'mean'),
            units_received_daily=('units_received', 'sum'),
            closing_stock_daily=('closing_stock', 'mean'),
        )
        .reset_index()
        .rename(columns={'date': 'effective_decision_date'})
    )
    base = complete_daily_feature_frame(store_df, categories, start, end)
    out = base.merge(daily, on=['store_id', 'category', 'effective_decision_date'], how='left')
    raw_cols = ['any_stockout', 'stockout_sku_share_daily', 'opening_stock_daily', 'units_received_daily', 'closing_stock_daily']
    out[raw_cols] = out[raw_cols].fillna(0)
    group_cols = ['store_id', 'category']
    out['stockout_rate_7'] = shifted_rolling_by_group(out, group_cols, 'any_stockout', 7, 'mean')
    out['stockout_rate_28'] = shifted_rolling_by_group(out, group_cols, 'any_stockout', 28, 'mean')
    out['stockout_sku_share_28'] = shifted_rolling_by_group(out, group_cols, 'stockout_sku_share_daily', 28, 'mean')
    out['opening_stock_mean_7'] = shifted_rolling_by_group(out, group_cols, 'opening_stock_daily', 7, 'mean')
    out['units_received_sum_7'] = shifted_rolling_by_group(out, group_cols, 'units_received_daily', 7, 'sum')
    out['closing_stock_mean_7'] = shifted_rolling_by_group(out, group_cols, 'closing_stock_daily', 7, 'mean')
    pieces = []
    for _, part in out.groupby(group_cols, observed=True, sort=False):
        part = part.copy()
        prior_stockout_date = part['effective_decision_date'].where(part['any_stockout'].shift(1).fillna(0).gt(0)).ffill()
        part['days_since_last_stockout'] = (part['effective_decision_date'] - prior_stockout_date).dt.days.fillna(99).clip(0, 99)
        pieces.append(part)
    out = pd.concat(pieces, ignore_index=True)
    out[INVENTORY_FEATURES[:-1]] = out[INVENTORY_FEATURES[:-1]].replace([np.inf, -np.inf], np.nan).fillna(0)
    return out[['store_id', 'category', 'effective_decision_date'] + INVENTORY_FEATURES[:-1]]


# รวมการสร้าง as-of feature blocks ทุกกลุ่มให้เรียกใช้งานจุดเดียว
def build_all_asof_features():
    """เรียกสร้าง as-of feature tables ทุกกลุ่มตามลำดับ และคืนตารางที่พร้อม merge เข้า model frame"""
    target_daily = make_target_daily_full(training_target_panel, store_table, CATEGORY_ORDER, FEATURE_START, HISTORY_END_DATE)
    print('Building as-of sales features')
    sales_asof = build_sales_asof_features(target_daily)
    print('Building momentum features')
    momentum_asof = build_momentum_features(target_daily)
    print('Building order/customer features')
    order_customer_asof = build_order_customer_features(order_table, transaction_table, product_table, store_table, CATEGORY_ORDER, FEATURE_START, HISTORY_END_DATE)
    print('Building revenue/discount features')
    revenue_discount_asof = build_revenue_discount_features(order_table, transaction_table, product_table, store_table, CATEGORY_ORDER, FEATURE_START, HISTORY_END_DATE)
    print('Building inventory features')
    inventory_asof = build_inventory_features(inventory_table, product_table, store_table, CATEGORY_ORDER, FEATURE_START, HISTORY_END_DATE)
    return sales_asof, momentum_asof, order_customer_asof, revenue_discount_asof, inventory_asof


# Materialize feature blocks ทั้งหมดเพื่อใช้ใน model-frame assembly
submission_request_frame = make_submission_target_rows(submission_template)
store_static = add_store_static_features(store_table)
category_static = build_category_static_features(product_table)
calendar_lookup = build_calendar_lookup(calendar_table, FEATURE_START, SUBMISSION_END_DATE)
promotion_daily_features = make_promotion_daily_features(promotion_table, product_table)
local_event_density = build_local_event_density_features(local_event_table, store_table, FEATURE_START, SUBMISSION_END_DATE)
event_relevance_features = build_event_relevance_features(local_event_table, store_table, CATEGORY_ORDER)
curated_event_calendar = make_curated_event_calendar()
special_event_feature_table = make_special_event_features(curated_event_calendar, store_table, CATEGORY_ORDER)
weather_features = build_weather_features(FEATURE_START, SUBMISSION_END_DATE)
(
    sales_asof_features,
    momentum_asof_features,
    order_customer_asof_features,
    revenue_discount_asof_features,
    inventory_asof_features,
) = build_all_asof_features()

print({
    'store_static': store_static.shape,
    'category_static': category_static.shape,
    'calendar_lookup': calendar_lookup.shape,
    'promo_features': promotion_daily_features.shape,
    'local_event_density': local_event_density.shape,
    'event_relevance_features': event_relevance_features.shape,
    'special_features': special_event_feature_table.shape,
    'weather_features': weather_features.shape,
    'sales_asof_features': sales_asof_features.shape,
    'momentum_asof_features': momentum_asof_features.shape,
    'order_customer_asof_features': order_customer_asof_features.shape,
    'revenue_discount_asof_features': revenue_discount_asof_features.shape,
    'inventory_asof_features': inventory_asof_features.shape,
})

## ขั้นที่ 7: รวม Feature ทั้งหมด

In [ ]:
# ส่วนนี้รวม feature blocks ทั้งหมดเป็น forecast_feature_frame พร้อม merge guards และ type finalization
# Helper กลุ่มนี้ทำให้ merge ปลอดภัยขึ้นโดยตรวจ duplicate keys และ row count
def key_list(keys):
    """แปลง key ที่รับเป็น string หรือ list ให้เป็น list เสมอ เพื่อให้ helper merge ใช้งานได้สม่ำเสมอ"""
    return [keys] if isinstance(keys, str) else list(keys)


def require_unique_keys(df, keys, name):
    """ตรวจว่า dataframe ฝั่งขวาของ merge ไม่มี duplicate keys เพื่อกัน row explosion แบบเงียบ"""
    keys = key_list(keys)
    duplicate_count = int(df.duplicated(keys).sum())
    if duplicate_count:
        examples = df.loc[df.duplicated(keys, keep=False), keys].head(5).to_dict('records')
        raise AssertionError(f'{name} has {duplicate_count} duplicate key rows on {keys}: {examples}')


def safe_left_merge(left, right, keys, name, columns=None):
    """ทำ left merge แบบมี guard โดยตรวจ unique keys และตรวจว่า row count ไม่เปลี่ยนหลัง merge"""
    keys = key_list(keys)
    right_view = right[keys + columns].copy() if columns is not None else right.copy()
    require_unique_keys(right_view, keys, name)
    before = len(left)
    merged = left.merge(right_view, on=keys, how='left', validate='many_to_one')
    if len(merged) != before:
        raise AssertionError(f'{name} changed row count from {before} to {len(merged)}')
    return merged


# Helper กลุ่มนี้กำหนดลำดับการ merge feature blocks เข้ากับ frame หลัก
def merge_feature_tables(frame):
    """merge feature tables ทุกกลุ่มเข้ากับ model frame ด้วย safe_left_merge ตาม key ของแต่ละ block"""
    feature_tables = [
        ('promo_features', promotion_daily_features, ['store_id', 'category', 'timestamp']),
        ('local_event_density', local_event_density, ['store_id', 'timestamp']),
        ('event_relevance_features', event_relevance_features, ['store_id', 'category', 'timestamp']),
        ('special_features', special_event_feature_table, ['store_id', 'category', 'timestamp']),
        ('weather_features', weather_features, ['timestamp']),
        ('sales_asof_features', sales_asof_features, ['store_id', 'category', 'effective_decision_date']),
        ('momentum_asof_features', momentum_asof_features, ['store_id', 'category', 'effective_decision_date']),
        ('order_customer_asof_features', order_customer_asof_features, ['store_id', 'category', 'effective_decision_date']),
        ('revenue_discount_asof_features', revenue_discount_asof_features, ['store_id', 'category', 'effective_decision_date']),
        ('inventory_asof_features', inventory_asof_features, ['store_id', 'category', 'effective_decision_date']),
    ]
    for name, table, keys in feature_tables:
        frame = safe_left_merge(frame, table, keys, name)
    return frame


# Helper กลุ่มนี้สร้าง derived features และ finalize dtype/missing values
def add_derived_model_features(frame):
    """สร้าง feature ที่คำนวณต่อจาก feature blocks เช่น stock cover proxy และอายุร้าน"""
    frame['stock_cover_proxy_28'] = frame['closing_stock_mean_7'] / (frame['decision_sales_mean_28'] + EPS)
    frame['stock_cover_proxy_28'] = frame['stock_cover_proxy_28'].replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 999)
    frame['store_age_days'] = (frame['timestamp'] - frame['opened_date']).dt.days.clip(lower=0)
    frame['store_age_months'] = frame['store_age_days'] / 30.4375
    frame['store_age_log'] = np.log1p(frame['store_age_days'])
    frame['is_new_store_365d'] = frame['store_age_days'].le(365).astype(int)
    return frame


def finalize_model_frame(frame):
    """ตรวจ feature columns, cast categorical columns และเติม missing numeric values ก่อนส่ง frame เข้าโมเดล"""
    missing_features = [c for c in model_feature_cols if c not in frame.columns]
    if missing_features:
        raise ValueError(f'Missing engineered features: {missing_features}')
    for col in ['category', 'horizon', 'neighborhood_type', 'day_of_week']:
        if col in frame.columns:
            frame[col] = frame[col].astype('category')
    frame['category'] = pd.Categorical(frame['category'].astype(str), CATEGORY_ORDER, ordered=True)
    frame['horizon'] = pd.Categorical(frame['horizon'].astype(str), list(HORIZON_DAY_LOOKUP.keys()), ordered=True)
    frame['day_of_week'] = pd.Categorical(frame['day_of_week'].astype(str), WEEKDAY_ORDER, ordered=True)
    frame['has_drive_through'] = frame['has_drive_through'].fillna(False).astype(int)
    numeric_feature_cols = [
        c for c in model_feature_cols
        if c not in ['store_id', 'category', 'horizon', 'neighborhood_type', 'day_of_week']
    ]
    for col in numeric_feature_cols:
        frame[col] = pd.to_numeric(frame[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    return frame.copy()


# ฟังก์ชันหลักของ cell นี้ ประกอบ index, target, static, calendar และ feature blocks เข้าด้วยกัน
def assemble_model_frame(history_cutoff, frame_start=FEATURE_START, frame_end=SUBMISSION_END_DATE):
    """ประกอบ forecast index, target และ feature blocks ทั้งหมดเป็น forecast_feature_frame ที่ใช้ train และ predict"""
    cutoff = pd.Timestamp(history_cutoff)
    frame = create_forecast_index_frame(store_table, CATEGORY_ORDER, HORIZON_DAY_LOOKUP, frame_start, frame_end, cutoff)
    assert frame['effective_decision_date'].max() <= cutoff
    frame = safe_left_merge(frame, training_target_panel[['store_id', 'category', 'timestamp', 'units_sold']], ['store_id', 'category', 'timestamp'], 'training_target_panel')
    frame = safe_left_merge(frame, store_static, 'store_id', 'store_static')
    frame = safe_left_merge(frame, category_static, 'category', 'category_static')
    require_unique_keys(calendar_lookup, 'timestamp', 'calendar_lookup')
    before_calendar = len(frame)
    frame = add_calendar_features(frame, calendar_lookup)
    frame = attach_decision_calendar_features(frame, calendar_lookup)
    if len(frame) != before_calendar:
        raise AssertionError(f'calendar features changed row count from {before_calendar} to {len(frame)}')
    frame = merge_feature_tables(frame)
    frame = add_derived_model_features(frame)
    return finalize_model_frame(frame)


# สร้าง forecast_feature_frame สำหรับ final training/future prediction และตรวจ feature contract
forecast_feature_frame = assemble_model_frame(history_cutoff=HISTORY_END_DATE, frame_end=SUBMISSION_END_DATE)
print('Known covariates:', len(future_known_feature_cols))
print('Static feature cols:', len(STATIC_FEATURE_COLS))
print('Tabular feature cols:', len(model_feature_cols))
print('Total model features:', len(STATIC_FEATURE_COLS) + len(future_known_feature_cols))
print('Model frame:', forecast_feature_frame.shape)
print('Missing values in features:', int(forecast_feature_frame[model_feature_cols].isna().sum().sum()))
forecast_feature_frame.head()

## ขั้นที่ 8: ทำ Backtest, Train Model และเลือก Ensemble

In [ ]:
# ส่วนนี้นิยาม model adapters, backtest runner, ensemble, calibration และ cap selection
# กลุ่มฟังก์ชันนี้เป็น AutoGluon adapter สำหรับแปลงข้อมูล train/predict
def make_timeseries_dataset(frame, include_target=True):
    """แปลง dataframe เป็น TimeSeriesDataFrame ของ AutoGluon โดยเลือก target และ known covariates ตามโหมดที่ใช้"""
    cols = ['item_id', 'timestamp'] + (['units_sold'] if include_target else []) + future_known_feature_cols
    return TimeSeriesDataFrame.from_data_frame(frame[cols], id_column='item_id', timestamp_column='timestamp')


def make_static_metadata(frame):
    """สร้าง static feature table ต่อ item_id สำหรับแนบเข้า AutoGluon TimeSeriesDataFrame"""
    static = frame.groupby('item_id', observed=True).agg({col: 'first' for col in STATIC_FEATURE_COLS})
    for col in ['store_id', 'category', 'horizon', 'neighborhood_type']:
        static[col] = static[col].astype(str).astype('category')
    return static


def fit_timeseries_predictor(training_frame, cutoff, model_path, time_limit):
    """train AutoGluon TimeSeriesPredictor ด้วยข้อมูลจนถึง cutoff และ known covariates ที่กำหนด"""
    model_path = Path(model_path)
    model_path.parent.mkdir(parents=True, exist_ok=True)
    observed = training_frame[training_frame['timestamp'].le(cutoff) & training_frame['units_sold'].notna()].copy()
    ts_data = make_timeseries_dataset(observed, include_target=True)
    ts_data.static_features = make_static_metadata(training_frame)
    fit_args = {
        'train_data': ts_data,
        'time_limit': time_limit,
        'hyperparameters': AUTOGLUON_MODEL_CONFIG,
        'num_val_windows': 2,
        'refit_every_n_windows': 1,
    }
    predictor = TimeSeriesPredictor(
        target='units_sold',
        prediction_length=FORECAST_WINDOW_DAYS,
        freq='D',
        eval_metric='MAE',
        known_covariates_names=future_known_feature_cols,
        path=str(model_path),
    )
    return predictor.fit(**fit_args)


# กำหนดชื่อ output point forecasts ที่ดึงจาก AutoGluon
AG_POINT_COLUMNS = {'ag_mean': 'mean', 'ag_p40': '0.4', 'ag_p50': '0.5', 'ag_p60': '0.6'}


def collect_ag_point_forecasts(predictor, frame, requested_rows, label='ag'):
    """ดึง point forecasts จาก AutoGluon สำหรับ decision cutoffs หลายวัน แล้ว align กลับตามลำดับ request rows"""
    requests = requested_rows.copy().reset_index(drop=True)
    requests['_request_order'] = np.arange(len(requests))
    collected = []
    decision_dates = sorted(pd.to_datetime(requests['effective_decision_date']).unique())
    print(f'{label}: {len(requests)} rows across {len(decision_dates)} decision cutoffs')
    for decision_date in decision_dates:
        decision_date = pd.Timestamp(decision_date)
        subset = requests[requests['effective_decision_date'].eq(decision_date)]
        history = frame[frame['timestamp'].le(decision_date) & frame['units_sold'].notna()].copy()
        future_end = decision_date + pd.Timedelta(days=FORECAST_WINDOW_DAYS)
        future = frame[frame['timestamp'].gt(decision_date) & frame['timestamp'].le(future_end)].copy()
        history_ts = make_timeseries_dataset(history, include_target=True)
        history_ts.static_features = make_static_metadata(frame)
        future_ts = make_timeseries_dataset(future, include_target=False)
        raw = predictor.predict(history_ts, known_covariates=future_ts).reset_index()
        for output_col, source_col in AG_POINT_COLUMNS.items():
            if source_col not in raw.columns and source_col != 'mean':
                raw[source_col] = raw['mean']
        aligned = subset[['_request_order', 'item_id', 'timestamp']].merge(
            raw[['item_id', 'timestamp'] + list(AG_POINT_COLUMNS.values())].rename(columns={v: k for k, v in AG_POINT_COLUMNS.items()}),
            on=['item_id', 'timestamp'],
            how='left',
        )
        collected.append(aligned)
        print({'cutoff': str(decision_date.date()), 'rows': len(subset), 'missing': int(aligned[list(AG_POINT_COLUMNS)].isna().sum().sum())})
    out = pd.concat(collected, ignore_index=True).sort_values('_request_order')
    for col in AG_POINT_COLUMNS:
        out[col] = out[col].clip(lower=0)
    return out[list(AG_POINT_COLUMNS)].reset_index(drop=True)


# กลุ่มฟังก์ชันนี้เป็น CatBoost adapter สำหรับ tabular features
CATBOOST_CATEGORICAL_FEATURES = {'store_id', 'category', 'horizon', 'neighborhood_type', 'day_of_week'}


def split_tabular_features_by_type(df, feature_cols):
    """เตรียม feature matrix ให้ CatBoost โดยแยก categorical columns และแปลง numeric columns เป็น float"""
    X = df[feature_cols].copy()
    cat_cols = sorted(CATBOOST_CATEGORICAL_FEATURES.intersection(X.columns).union(X.select_dtypes(include=['object', 'category', 'bool']).columns))
    for col in cat_cols:
        X[col] = X[col].astype(str).fillna('missing')
    for col in X.columns.difference(cat_cols):
        X[col] = pd.to_numeric(X[col], errors='coerce').astype('float32')
    cat_idx = [X.columns.get_loc(col) for col in cat_cols]
    return X, cat_idx


def fit_catboost_model_recipe(train_rows, recipe_name, iterations):
    """train CatBoost ตามสูตรที่กำหนด ทั้งแบบ target raw และ log เพื่อเป็น candidate ใน ensemble"""
    X_train, cat_idx = split_tabular_features_by_type(train_rows, model_feature_cols)
    y_raw = train_rows['units_sold'].astype(float).clip(lower=0).values
    if recipe_name == 'catboost_log1p':
        y_train, learning_rate, l2_leaf_reg, seed_offset = np.log1p(y_raw), 0.043, 6.0, 0
    elif recipe_name == 'catboost_raw':
        y_train, learning_rate, l2_leaf_reg, seed_offset = y_raw, 0.038, 8.0, 11
    else:
        raise ValueError(recipe_name)
    model = CatBoostRegressor(
        loss_function='MAE',
        iterations=iterations,
        learning_rate=learning_rate,
        depth=6,
        l2_leaf_reg=l2_leaf_reg,
        random_seed=SEED + seed_offset,
        thread_count=-1,
        verbose=False,
        allow_writing_files=False,
    )
    model.fit(X_train, y_train, cat_features=cat_idx)
    return model


def predict_catboost_model_recipe(model, rows, recipe_name):
    """สร้าง prediction จาก CatBoost model และแปลงกลับจาก log scale"""
    X_rows, _ = split_tabular_features_by_type(rows, model_feature_cols)
    pred = model.predict(X_rows)
    if recipe_name == 'catboost_log1p':
        pred = np.expm1(pred)
    return np.clip(pred, 0, None)



# กลุ่มฟังก์ชันนี้เลือก blend weights และประเมิน weighted predictions
def fit_simplex_blend(oof_df, prediction_cols, n_iter=BLEND_RANDOM_SEARCH_ITER, seed=SEED):
    """ค้นหาน้ำหนัก blend แบบ non-negative simplex จาก OOF predictions เพื่อลด MAE บน validation"""
    y = oof_df['units_sold'].astype(float).values
    preds = np.vstack([oof_df[col].values for col in prediction_cols])
    rng = np.random.default_rng(seed)
    best_mae = np.inf
    best_weights = None
    candidates = []
    for i in range(len(prediction_cols)):
        weights = np.zeros(len(prediction_cols))
        weights[i] = 1
        candidates.append(weights)
    for i in range(len(prediction_cols)):
        for j in range(i + 1, len(prediction_cols)):
            for wi in np.arange(0, 1.0001, 0.025):
                weights = np.zeros(len(prediction_cols))
                weights[i], weights[j] = wi, 1 - wi
                candidates.append(weights)
    candidates.extend(rng.dirichlet(np.ones(len(prediction_cols)) * 0.7) for _ in range(n_iter))
    for weights in candidates:
        mae = mean_absolute_error(y, (weights[:, None] * preds).sum(axis=0))
        if mae < best_mae:
            best_mae, best_weights = mae, weights.copy()
    return dict(zip(prediction_cols, best_weights)), best_mae


def evaluate_blend_candidates(df, weight_dict):
    """คำนวณ weighted average prediction จาก candidate columns ตาม blend weights ที่เลือกไว้"""
    pred = np.zeros(len(df), dtype=float)
    for col, weight in weight_dict.items():
        pred += float(weight) * df[col].values
    return np.clip(pred, 0, None)


# กลุ่มฟังก์ชันนี้ทำ residual calibration แบบ group-aware
def fit_bias_ratio_adjuster(train_rows, pred_col, group_cols, shrink_n=CALIBRATION_SHRINK_N):
    """fit calibration table จาก residual และ ratio ของ prediction ต่อ target ตามกลุ่มที่กำหนด"""
    work = train_rows.copy()
    if not group_cols:
        work['__global_key'] = 'global'
        group_cols = ['__global_key']
    work['_ratio'] = (work['units_sold'].astype(float) + 1) / (work[pred_col].astype(float) + 1)
    work['_bias'] = work['units_sold'].astype(float) - work[pred_col].astype(float)
    table = work.groupby(group_cols, observed=True).agg(n=('units_sold', 'size'), factor_raw=('_ratio', 'median'), bias_raw=('_bias', 'median')).reset_index()
    table['shrink'] = table['n'] / (table['n'] + shrink_n)
    table['factor_raw'] = table['factor_raw'].replace([np.inf, -np.inf], np.nan).fillna(1.0).clip(0.90, 1.10)
    table['bias_raw'] = table['bias_raw'].replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-6.0, 6.0)
    table['factor'] = 1.0 + (table['factor_raw'] - 1.0) * table['shrink']
    table['bias'] = table['bias_raw'] * table['shrink']
    return table, group_cols


def apply_bias_ratio_adjuster(rows, pred_col, table, group_cols):
    """นำ calibration factor และ bias ไปปรับ prediction ของแถวใหม่ตาม group columns"""
    work = rows.copy()
    if group_cols == ['__global_key']:
        work['__global_key'] = 'global'
    merged = work.merge(table[group_cols + ['factor', 'bias']], on=group_cols, how='left')
    return np.clip(merged[pred_col].values * merged['factor'].fillna(1.0).values + merged['bias'].fillna(0.0).values, 0, None)


def select_residual_adjustment(oof_df, pred_col):
    """เลือก schema calibration ที่ดีที่สุดจาก global, horizon, category และ category+horizon ด้วย fold-aware validation"""
    choices = [[], ['horizon'], ['category'], ['category', 'horizon']]
    rows, best = [], None
    for group_cols in choices:
        pieces = []
        for fold in sorted(oof_df['fold'].unique()):
            train_part = oof_df[oof_df['fold'].ne(fold)].copy()
            valid_part = oof_df[oof_df['fold'].eq(fold)].copy()
            table, actual_groups = fit_bias_ratio_adjuster(train_part, pred_col, group_cols)
            pred = apply_bias_ratio_adjuster(valid_part, pred_col, table, actual_groups)
            pieces.append(valid_part[['fold', 'category', 'horizon', 'timestamp', 'units_sold']].assign(adjusted_prediction=pred))
        combined = pd.concat(pieces, ignore_index=True)
        mae = mean_absolute_error(combined['units_sold'], combined['adjusted_prediction'])
        name = 'global' if not group_cols else '+'.join(group_cols)
        rows.append({'calibration': name, 'mae': mae})
        if best is None or mae < best['mae']:
            best = {'calibration': name, 'group_cols': group_cols, 'mae': mae}
    return best, pd.DataFrame(rows).sort_values('mae').reset_index(drop=True)


# กลุ่มฟังก์ชันนี้สร้างและเลือก cap policy เพื่อลด prediction ที่สูงผิดปกติ
def build_store_category_caps(cutoff, q=0.995, multiplier=1.30):
    """สร้าง upper cap ต่อ store/category จากประวัติยอดขาย เพื่อจำกัด prediction ที่สูงผิดปกติ"""
    hist = training_target_panel[training_target_panel['timestamp'].le(pd.Timestamp(cutoff))].copy()
    cap = hist.groupby(['store_id', 'category'], observed=True)['units_sold'].agg(
        hist_mean='mean',
        hist_p995=lambda s: s.quantile(q),
        hist_max='max',
    ).reset_index()
    cap['prediction_cap'] = np.maximum(cap['hist_p995'] * multiplier, cap['hist_mean'] + 3.25 * np.sqrt(cap['hist_mean'] + 1))
    cap['prediction_cap'] = np.maximum(cap['prediction_cap'], cap['hist_max'] * 0.72)
    return cap[['store_id', 'category', 'prediction_cap']]


def apply_cap_policy(rows, predictions, fold_cutoffs, multiplier):
    """apply cap policy ให้ predictions โดยใช้ cap table ที่สร้างจาก cutoff ของแต่ละ fold"""
    capped = np.zeros(len(rows), dtype=float)
    work = rows[['store_id', 'category', 'fold']].copy()
    work['_prediction'] = predictions
    for fold, idx in work.groupby('fold').groups.items():
        cap_table = build_store_category_caps(fold_cutoffs[fold], multiplier=multiplier)
        part = work.loc[idx, ['store_id', 'category', '_prediction']].merge(cap_table, on=['store_id', 'category'], how='left')
        capped[np.asarray(list(idx))] = np.minimum(part['_prediction'].values, part['prediction_cap'].fillna(np.inf).values)
    return capped


def choose_cap_policy(oof_df, pred_col, fold_cutoffs):
    """เลือก cap multiplier จาก grid search โดยดู MAE หลัง cap บน OOF validation"""
    rows = []
    for multiplier in CAP_TUNE_GRID_MULTS:
        capped = apply_cap_policy(oof_df, oof_df[pred_col].values, fold_cutoffs, multiplier)
        rows.append({'cap_mult': multiplier, 'mae': mean_absolute_error(oof_df['units_sold'], capped)})
    scores = pd.DataFrame(rows).sort_values('mae').reset_index(drop=True)
    return float(scores.iloc[0]['cap_mult']), scores


# กลุ่มฟังก์ชันนี้รัน time-based backtest และรวม OOF predictions
def run_backtest_fold(fold_spec):
    """รัน backtest หนึ่ง fold ตั้งแต่สร้าง frame, train โมเดล, predict และคืน OOF predictions"""
    fold_name = fold_spec['name']
    cutoff = fold_spec['cutoff']
    val_start = fold_spec['valid_start']
    val_end = fold_spec['valid_end']
    print(f'Running {fold_name}: cutoff={cutoff.date()} valid={val_start.date()}..{val_end.date()}')
    fold_frame = assemble_model_frame(history_cutoff=cutoff, frame_end=max(val_end, cutoff + pd.Timedelta(days=FORECAST_WINDOW_DAYS)))
    val_rows = fold_frame[fold_frame['timestamp'].between(val_start, val_end) & fold_frame['units_sold'].notna()].copy().reset_index(drop=True)
    train_rows = fold_frame[fold_frame['timestamp'].le(cutoff) & fold_frame['units_sold'].notna()].copy().reset_index(drop=True)
    ag_predictor = fit_timeseries_predictor(fold_frame, cutoff, MODEL_ARTIFACT_DIR / f'ag_teaching_{fold_name}', AUTOGLUON_CV_TIME_LIMIT_SECONDS)
    ag_points = collect_ag_point_forecasts(ag_predictor, fold_frame, val_rows, label=f'ag_{fold_name}')
    cb_log = fit_catboost_model_recipe(train_rows, 'catboost_log1p', CATBOOST_ITERATIONS_CV)
    cb_raw = fit_catboost_model_recipe(train_rows, 'catboost_raw', CATBOOST_ITERATIONS_CV)
    fold_oof = val_rows[['store_id', 'category', 'horizon', 'timestamp', 'units_sold', 'effective_decision_date']].copy()
    fold_oof['fold'] = fold_name
    for col in ag_points.columns:
        fold_oof[col] = ag_points[col].values
    fold_oof['catboost_log1p'] = predict_catboost_model_recipe(cb_log, val_rows, 'catboost_log1p')
    fold_oof['catboost_raw'] = predict_catboost_model_recipe(cb_raw, val_rows, 'catboost_raw')
    return {'fold': fold_name, 'cutoff': cutoff, 'oof': fold_oof}


def run_all_backtests(fold_specs):
    """รัน backtest ทุก fold แล้วรวม OOF predictions และ cutoff map สำหรับ tuning ensemble/postprocess"""
    results = [run_backtest_fold(fold) for fold in fold_specs]
    return {
        'oof': pd.concat([result['oof'] for result in results], ignore_index=True),
        'fold_cutoffs': {result['fold']: result['cutoff'] for result in results},
    }


# เลือก ensemble/postprocess จาก CV หรือใช้ fallback weights เมื่อปิด CV
if ENABLE_BACKTEST:
    backtest_result = run_all_backtests(BACKTEST_WINDOWS)
    backtest_oof_predictions = backtest_result['oof']
    backtest_cutoff_by_fold = backtest_result['fold_cutoffs']
    pred_cols = list(AG_POINT_COLUMNS) + ['catboost_log1p', 'catboost_raw']
    ensemble_weight_map, _ = fit_simplex_blend(backtest_oof_predictions, pred_cols)
    backtest_oof_predictions['multi_blend_pred'] = evaluate_blend_candidates(backtest_oof_predictions, ensemble_weight_map)
    calib_best, _ = select_residual_adjustment(backtest_oof_predictions, 'multi_blend_pred')
    base_blend_mae = mean_absolute_error(backtest_oof_predictions['units_sold'], backtest_oof_predictions['multi_blend_pred'])
    if calib_best['mae'] < base_blend_mae:
        selected_calibration_table, selected_calibration_keys = fit_bias_ratio_adjuster(backtest_oof_predictions, 'multi_blend_pred', calib_best['group_cols'])
        backtest_oof_predictions['multi_blend_calibrated_pred'] = apply_bias_ratio_adjuster(backtest_oof_predictions, 'multi_blend_pred', selected_calibration_table, selected_calibration_keys)
        chosen_prediction_column = 'multi_blend_calibrated_pred'
    else:
        selected_calibration_table, selected_calibration_keys = pd.DataFrame(), []
        backtest_oof_predictions['multi_blend_calibrated_pred'] = backtest_oof_predictions['multi_blend_pred']
        chosen_prediction_column = 'multi_blend_pred'
    selected_cap_multiplier, _ = choose_cap_policy(backtest_oof_predictions, chosen_prediction_column, backtest_cutoff_by_fold)
    backtest_oof_predictions['selected_capped_pred'] = apply_cap_policy(backtest_oof_predictions, backtest_oof_predictions[chosen_prediction_column].values, backtest_cutoff_by_fold, selected_cap_multiplier)
    print({
        'blend_mae': base_blend_mae,
        'selected_mae': mean_absolute_error(backtest_oof_predictions['units_sold'], backtest_oof_predictions['selected_capped_pred']),
        'cap_multiplier': selected_cap_multiplier,
        'ensemble_weight_map': ensemble_weight_map,
    })
else:
    pred_cols = list(AG_POINT_COLUMNS) + ['catboost_log1p', 'catboost_raw']
    ensemble_weight_map = {'ag_mean': 0.25, 'ag_p40': 0.0, 'ag_p50': 0.25, 'ag_p60': 0.0, 'catboost_log1p': 0.35, 'catboost_raw': 0.15}
    selected_calibration_table, selected_calibration_keys = pd.DataFrame(), []
    chosen_prediction_column = 'multi_blend_pred'
    selected_cap_multiplier = 1.30
    print('ENABLE_BACKTEST=False fallback weights:', ensemble_weight_map)

## ขั้นที่ 9: Train Final Model, Predict และเขียน Submission

In [ ]:
# ส่วนนี้ train final models, predict sample rows และเขียน submission ไฟล์เดียว
# กลุ่มฟังก์ชันนี้ดูแล final model training, final prediction และ final postprocess
def train_final_models():
    """train final AutoGluon และ CatBoost models ด้วยข้อมูลอดีตทั้งหมดจนถึง HISTORY_END_DATE เพื่อใช้สร้าง submission"""
    if RUN_FINAL_AUTOGLUON_TRAINING:
        ag_model = fit_timeseries_predictor(forecast_feature_frame, HISTORY_END_DATE, MODEL_ARTIFACT_DIR / 'ag_final_teaching', AUTOGLUON_FINAL_TIME_LIMIT_SECONDS)
        print(ag_model.leaderboard())
    else:
        ag_model = TimeSeriesPredictor.load(str(MODEL_ARTIFACT_DIR / 'ag_final_teaching'))
    final_train_rows = forecast_feature_frame[forecast_feature_frame['timestamp'].le(HISTORY_END_DATE) & forecast_feature_frame['units_sold'].notna()].copy().reset_index(drop=True)
    cb_log_model = fit_catboost_model_recipe(final_train_rows, 'catboost_log1p', CATBOOST_ITERATIONS_FINAL)
    cb_raw_model = fit_catboost_model_recipe(final_train_rows, 'catboost_raw', CATBOOST_ITERATIONS_FINAL)
    return {'ag': ag_model, 'catboost_log1p': cb_log_model, 'catboost_raw': cb_raw_model}


def predict_final_components(models, target_rows):
    """สร้าง component predictions สำหรับ sample rows จาก AutoGluon และ CatBoost ทุกสูตร"""
    ag_points = collect_ag_point_forecasts(models['ag'], forecast_feature_frame, target_rows, label='ag_final')
    out = target_rows[['id', 'store_id', 'category', 'horizon', 'timestamp']].rename(columns={'timestamp': 'forecast_date'}).copy()
    for col in ag_points.columns:
        out[col] = ag_points[col].values
    out['catboost_log1p'] = predict_catboost_model_recipe(models['catboost_log1p'], target_rows, 'catboost_log1p')
    out['catboost_raw'] = predict_catboost_model_recipe(models['catboost_raw'], target_rows, 'catboost_raw')
    return out


def apply_final_blend_and_cap(component_frame):
    """นำ final component predictions ไป blend, calibrate และ cap เพื่อได้ prediction สุดท้ายสำหรับ submission"""
    out = component_frame.copy()
    out['multi_blend'] = evaluate_blend_candidates(out, ensemble_weight_map)
    if len(selected_calibration_table):
        out['multi_blend_calibrated'] = apply_bias_ratio_adjuster(
            out.rename(columns={'forecast_date': 'timestamp'}),
            'multi_blend',
            selected_calibration_table,
            selected_calibration_keys,
        )
    else:
        out['multi_blend_calibrated'] = out['multi_blend'].values
    cap_table = build_store_category_caps(HISTORY_END_DATE, multiplier=selected_cap_multiplier)
    caps = out[['store_id', 'category']].merge(cap_table, on=['store_id', 'category'], how='left')['prediction_cap'].fillna(np.inf).values
    out['selected_capped'] = np.minimum(out['multi_blend_calibrated'].values, caps)
    return out


def write_single_submission_file(ids, predictions, filename='submission.csv'):
    """เขียนไฟล์ submission สุดท้ายหลังตรวจค่า prediction และปัดยอดขายให้เป็นจำนวนเต็ม"""
    raw_predictions = np.asarray(predictions, dtype=float)
    assert np.isfinite(raw_predictions).all()
    clipped_predictions = np.clip(raw_predictions, 0, None)
    rounded_predictions = np.ceil(clipped_predictions).astype(np.int64)
    sub = pd.DataFrame({'id': ids.values, 'units_sold_predicted': rounded_predictions})
    assert len(sub) == len(ids)
    assert sub['units_sold_predicted'].notna().all()
    assert (sub['units_sold_predicted'] >= 0).all()
    assert pd.api.types.is_integer_dtype(sub['units_sold_predicted'])
    output_path = WORKSPACE_DIR / filename
    sub.to_csv(output_path, index=False)
    print('Wrote final submission:', output_path)
    display(sub['units_sold_predicted'].describe(percentiles=[.01, .05, .1, .5, .9, .95, .99]))
    return sub


# Join sample rows เข้ากับ forecast_feature_frame และตรวจว่า features สำหรับ submission ครบทุกแถว
submission_model_rows = submission_request_frame.rename(columns={'forecast_date': 'timestamp'}).merge(
    forecast_feature_frame,
    on=['store_id', 'category', 'horizon', 'horizon_days', 'timestamp', 'decision_date', 'effective_decision_date', 'item_id'],
    how='left',
    suffixes=('', '_frame'),
    validate='one_to_one',
)
assert len(submission_model_rows) == len(submission_template)
assert submission_model_rows['id'].reset_index(drop=True).equals(submission_template['id'].reset_index(drop=True))
missing_sample_features = int(submission_model_rows[model_feature_cols].isna().sum().sum())
if missing_sample_features:
    raise AssertionError(f'Sample rows have {missing_sample_features} missing feature values after model-frame join')

# เริ่ม final inference pipeline แล้วเขียน submission สุดท้าย
final_models = train_final_models()
final_components = predict_final_components(final_models, submission_model_rows)
final_prediction_frame = apply_final_blend_and_cap(final_components)
write_single_submission_file(submission_template['id'], final_prediction_frame['selected_capped'])
final_prediction_frame.head()